# Week 12 Task 4.2 - 路线识别一体化 Notebook

**这个 notebook 是自包含的。** 迷宫识别、墙体检测、栅格规划、5x5 连续避障规划、
路线绘图、Arduino 指令生成全部写在下面的单元格里，不再 `import continuous_planner`,
也不再需要 `grid_wall_detector.py`。示例照片以 base64 内嵌，所以把这一个 `.ipynb`
文件发给别人，对方打开就能直接跑。

## 怎么用

1. 从上到下 **Run All**（或按顺序执行）。第 1 节会检查依赖。
2. 在 **第 2 节** 改三样东西：照片来源、5x5 场地位置、起点终点。
3. 第 **13 节**做识别和规划，第 **14 节**画出路线图并打印可以直接粘贴进
   Arduino 的 `#define GENERATED_ROUTE "..."` 那一行。

## 照片来源有三种

| `IMAGE_SOURCE` | 说明 |
| --- | --- |
| `"demo"` | 用内嵌的示例照片，什么都不用准备（默认） |
| `"camera"` | 直接调用电脑摄像头拍一张，支持切换外接摄像头，拍完自动保存并识别 |
| `"file"`   | 用 `IMAGE_PATH` 指定的本地照片 |

## 需要的第三方库

`numpy`、`opencv-python`、`matplotlib`。串口下发（可选，第 16 节）另外需要 `pyserial`。

> 一句话说明坐标：整个迷宫是 9 x 9 格、每格 180 mm，位姿写成 `(行, 列, 朝向)`，
> 行列都是 0 开始，朝向是 `"N" / "E" / "S" / "W"`。

## 1. 环境自检

先确认三个库都在。缺什么会直接把安装命令打出来，照着复制到终端跑一次即可。

In [ ]:
import importlib.util
import sys

# 模块名 -> pip 包名（两者并不总是一样，比如 cv2 对应 opencv-python）
REQUIRED = {"numpy": "numpy", "cv2": "opencv-python", "matplotlib": "matplotlib"}

missing = [pip_name for module, pip_name in REQUIRED.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("缺少依赖：", ", ".join(missing))
    print("请复制下面这行到终端执行，然后重启内核：")
    print(f"    {sys.executable} -m pip install " + " ".join(missing))
else:
    import matplotlib
    import numpy
    import cv2
    print("Python     ", sys.version.split()[0])
    print("numpy      ", numpy.__version__)
    print("opencv     ", cv2.__version__)
    print("matplotlib ", matplotlib.__version__)
    print("\n依赖齐全，可以继续。")

# 这套代码里的类型标注用了 `int | None` 写法。Python 3.10 以下需要靠
# `from __future__ import annotations` 把标注延后成字符串，后面每个代码单元
# 都加了这一行，所以 3.9 也能跑；但仍建议用 3.10 以上。
if sys.version_info < (3, 9):
    print("\n警告：建议使用 Python 3.9 以上。")

## 2. 参数：**只需要改这一格**

- `IMAGE_SOURCE`：`"demo"` 内嵌示例 / `"camera"` 拍照 / `"file"` 本地文件。
- `COURSE_TOP_ROW`、`COURSE_LEFT_COLUMN`：5x5 障碍场地左上角所在的格子。
  **默认 `None`，也就是自动从照片里找**（场地位置每张图都可能不一样）。
  场地的开口（门洞）位置和数量同样是自动识别的。
- `START`、`GOAL`：起点和终点位姿 `(行, 列, 朝向)`。两个都必须落在 5x5 场地
  **之外**的普通格子里。
- `CAMERA_INDEX`：`0` 一般是笔记本自带摄像头，外接 USB 摄像头通常是 `1` 或 `2`。
  拍照预览窗口里按 `N` 可以当场切换，不用回来改这个数。

In [ ]:
# ---- 照片来源 ----------------------------------------------------------
IMAGE_SOURCE = "demo"        # "demo" | "camera" | "file"
IMAGE_PATH = None            # IMAGE_SOURCE = "file" 时填，例如 r"C:\pics\maze.jpg"
CAMERA_INDEX = 0             # 0 = 自带摄像头，外接一般是 1 或 2
CAPTURE_DIR = "captures"     # 拍下来的照片存在这个文件夹

# ---- 5x5 障碍场地在 9x9 迷宫里的位置 -----------------------------------
# None = 自动从照片里找。场地位置每张图都可能不一样，所以默认自动。
# 只有自动识别失败、而你确定场地在哪里时，才手动填数字覆盖它。
COURSE_TOP_ROW = None
COURSE_LEFT_COLUMN = None

# ---- 起点 / 终点位姿 (行, 列, 朝向) ------------------------------------
START = (6, 8, "N")
GOAL = (2, 1, "N")

# ---- 输出 --------------------------------------------------------------
SAVE_OUTPUT_DIR = None       # None = 只在 notebook 里内联显示，不写文件
                             # 想存 PNG 就写成 "output"

print(f"照片来源 : {IMAGE_SOURCE}")
print(f"5x5 场地 : 左上角 (行 {COURSE_TOP_ROW}, 列 {COURSE_LEFT_COLUMN})")
print(f"起点     : {START}")
print(f"终点     : {GOAL}")

## 3. 内嵌示例照片

下面这个 base64 字符串就是 `pics/c10.jpg` 本身，所以这个 notebook 不依赖任何外部
图片文件。字符串很长，单元格折叠起来看即可。

In [ ]:
# 内嵌示例照片（pics/c10.jpg），共 221388 字节。
DEMO_IMAGE_BASE64 = (
    "..."
)

print(f"内嵌示例照片：{len(DEMO_IMAGE_BASE64)} 个 base64 字符")

## 4. 公共依赖与几何常量

`MAZE_CELLS = 9`、`CELL_MM = 180`，所以整个迷宫是 1620 x 1620 mm，后面所有坐标都
以毫米为单位、原点在左上角。`BLOCKED_CORNERS` 是标准板四个被切掉的角上的 12 个格子。

In [ ]:
from __future__ import annotations

import base64
import heapq
import math
import time
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Sequence

import cv2
import numpy as np

# ---- 迷宫几何 ----------------------------------------------------------
MAZE_CELLS = 9
COURSE_CELLS = 5
CELL_MM = 180
MAZE_MM = MAZE_CELLS * CELL_MM
DISPLAY_PX = 720
PANEL_PX = 420
DIRECTIONS = ("N", "E", "S", "W")
DELTAS = ((-1, 0), (0, 1), (1, 0), (0, -1))
DIRECTION_HEADINGS = (0, -90, 180, 90)
BLOCKED_CORNERS = (
    (0, 0), (0, 1), (1, 0),
    (0, 7), (0, 8), (1, 8),
    (7, 0), (8, 0), (8, 1),
    (7, 8), (8, 7), (8, 8),
)

## 5. 数据结构

- `PlannerConfig`：所有可调参数集中在这里（机器人尺寸、安全边界、5x5 场地位置等）。
- `GlobalMap`：一次识别的结果（矫正后的图、墙体掩膜、膨胀后的占据栅格、圆柱坐标）。
- `GridMaze`：把墙体整理成 9x9 的格子邻接关系，给栅格规划用。
- `HybridPlan`：最终方案（栅格段 + 5x5 连续段）。

> **注意** `wall_clearance_mm` 只有 35 mm、`cylinder_clearance_mm` 是 50+35=85 mm，
> 这两个膨胀量**没有把机器人自身半径算进去**（规划时把机器人当成一个点）。
> 车宽 76 mm、半宽 38 mm，所以最紧的规划线贴着柱子过时理论上会蹭到。
> 目前这一层余量是靠 Arduino 端的实时避障兜住的。要从源头解决，就把
> `cylinder_clearance_mm` 改成 `obstacle_diameter_mm / 2 + robot_radius_mm + cylinder_safety_mm`，
> 但要重新验证你的地图还有没有解。

In [ ]:
from __future__ import annotations

# ---- 5.1 规划参数与数据结构 ----
@dataclass(frozen=True)
class CoursePortal:
    side: str
    offset: int


@dataclass(frozen=True)
class PlannerConfig:
    robot_diameter_mm: float = 100.0
    obstacle_diameter_mm: float = 100.0
    wall_safety_mm: float = 35.0
    cylinder_safety_mm: float = 35.0
    grid_mm: int = 10
    minimum_course_segment_mm: int = 40
    maximum_segment_mm: int = 1000
    route_turn_penalty_mm: float = 120.0
    # None means "find the 5 x 5 course in the photograph". Set an integer to
    # pin it down by hand. Use resolve_course_config() before reading these.
    course_top_row: int | None = None
    course_left_column: int | None = None
    portal_open_threshold: float = 0.35
    start_heading_deg: int = 0
    goal_heading_deg: int | None = 0

    @property
    def robot_radius_mm(self) -> float:
        return self.robot_diameter_mm / 2

    @property
    def wall_clearance_mm(self) -> float:
        return self.wall_safety_mm

    @property
    def cylinder_clearance_mm(self) -> float:
        return self.obstacle_diameter_mm / 2 + self.cylinder_safety_mm


@dataclass(frozen=True)
class Motion:
    turn_deg: int
    distance_mm: int
    heading_deg: int


@dataclass
class GlobalMap:
    rectified: np.ndarray
    wall_mask: np.ndarray
    occupancy_mask: np.ndarray
    obstacles_mm: list[tuple[float, float]]
    dark_threshold: int
    horizontal_walls: np.ndarray
    vertical_walls: np.ndarray
    horizontal_scores: np.ndarray
    vertical_scores: np.ndarray
    # Where the 5 x 5 obstacle course was found in this photograph.
    course_top_row: int = 0
    course_left_column: int = 0
    course_candidates: tuple = ()


@dataclass
class PlanResult:
    map_data: GlobalMap
    config: PlannerConfig
    control_points_mm: list[tuple[float, float]]
    waypoints_mm: list[tuple[float, float]]
    motions: list[Motion]
    occupancy_image: np.ndarray
    route_image: np.ndarray
    start_heading_deg: int
    goal_heading_deg: int | None


@dataclass
class GridMaze:
    horizontal_walls: np.ndarray
    vertical_walls: np.ndarray
    blocked_cells: np.ndarray

    def can_move(self, row: int, column: int, direction: int) -> bool:
        if self.blocked_cells[row, column]:
            return False
        if direction == 0:
            wall = self.horizontal_walls[row, column]
        elif direction == 1:
            wall = self.vertical_walls[row, column + 1]
        elif direction == 2:
            wall = self.horizontal_walls[row + 1, column]
        else:
            wall = self.vertical_walls[row, column]
        next_row = row + DELTAS[direction][0]
        next_column = column + DELTAS[direction][1]
        return (
            not wall
            and 0 <= next_row < MAZE_CELLS
            and 0 <= next_column < MAZE_CELLS
            and not self.blocked_cells[next_row, next_column]
        )


@dataclass(frozen=True)
class PortalPose:
    portal: CoursePortal
    inside_cell: tuple[int, int]
    outside_cell: tuple[int, int]
    inward_direction: int

    @property
    def outward_direction(self) -> int:
        return (self.inward_direction + 2) % 4


@dataclass
class HybridPlan:
    map_data: GlobalMap
    grid_maze: GridMaze
    config: PlannerConfig
    start_pose: tuple[int, int, int]
    goal_pose: tuple[int, int, int]
    entry: PortalPose | None
    exit: PortalPose | None
    grid_before: str
    grid_after: str
    cells_before: list[tuple[int, int]]
    cells_after: list[tuple[int, int]]
    continuous: PlanResult | None
    occupancy_image: np.ndarray
    route_image: np.ndarray

    @property
    def uses_course(self) -> bool:
        return self.continuous is not None

## 6. 读图与透视矫正

拍下来的照片是斜的，第一步要把它掰正成正对着的 1620 x 1620 mm 俯视图：

1. `_cyan_mask` 找出板子四角的青色标记；
2. `_largest_cluster` 去掉离群点，`_maze_quad` 拟合出迷宫的四边形；
3. `_warp` 做透视变换；
4. `_fit_lattice` 再用投影峰值把栅格对齐，消掉最后几个毫米的偏差。

矫正之后，图上 1 个像素 = 1 mm，后面所有算法都在这个坐标系里工作。

In [ ]:
from __future__ import annotations

# ---- 6.1 读图、找角点、透视矫正 ----
def read_image(path: str | Path) -> np.ndarray:
    path = Path(path)
    image = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def write_image(path: str | Path, image: np.ndarray) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, encoded = cv2.imencode(path.suffix or ".png", image)
    if not ok:
        raise ValueError(f"Could not encode image: {path}")
    encoded.tofile(str(path))


def _order_quad(points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=np.float32)
    sums = points.sum(axis=1)
    differences = np.diff(points, axis=1).ravel()
    return np.array(
        [
            points[np.argmin(sums)],
            points[np.argmin(differences)],
            points[np.argmax(sums)],
            points[np.argmax(differences)],
        ],
        dtype=np.float32,
    )


def _cyan_mask(image: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (86, 55, 28), (108, 255, 255))
    return cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)),
    )


def _largest_cluster(points: np.ndarray, radius: float) -> np.ndarray:
    unused = set(range(len(points)))
    clusters: list[list[int]] = []
    radius_squared = radius * radius
    while unused:
        seed = unused.pop()
        cluster = [seed]
        queue = [seed]
        while queue:
            current = queue.pop()
            candidates = list(unused)
            if not candidates:
                continue
            delta = points[candidates] - points[current]
            nearby = [
                candidates[index]
                for index, value in enumerate(delta)
                if float(value @ value) <= radius_squared
            ]
            for index in nearby:
                unused.remove(index)
                cluster.append(index)
                queue.append(index)
        clusters.append(cluster)
    return points[max(clusters, key=len)]


def _maze_quad(image: np.ndarray) -> np.ndarray:
    height, width = image.shape[:2]
    scale = min(1.0, 1800.0 / max(height, width))
    small = cv2.resize(image, None, fx=scale, fy=scale) if scale < 1 else image.copy()
    cyan = _cyan_mask(small)
    count, _, stats, centroids = cv2.connectedComponentsWithStats(cyan)
    image_area = small.shape[0] * small.shape[1]
    points = np.array(
        [
            centroids[index]
            for index in range(1, count)
            if max(4, int(image_area * 0.000002))
            <= stats[index, cv2.CC_STAT_AREA]
            <= max(300, int(image_area * 0.004))
        ],
        dtype=np.float32,
    )
    if len(points) >= 20:
        cluster = _largest_cluster(points, 0.13 * min(small.shape[:2]))
        if len(cluster) >= 15:
            return _order_quad(cv2.boxPoints(cv2.minAreaRect(cluster)))

    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (7, 7), 0), 45, 130)
    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (11, 11)),
        iterations=2,
    )
    candidates: list[tuple[float, np.ndarray]] = []
    for contour in cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)[-2]:
        area = cv2.contourArea(contour)
        if area < image_area * 0.12:
            continue
        hull = cv2.convexHull(contour)
        perimeter = cv2.arcLength(hull, True)
        for epsilon in (0.015, 0.025, 0.04):
            polygon = cv2.approxPolyDP(hull, epsilon * perimeter, True)
            if len(polygon) == 4 and cv2.isContourConvex(polygon):
                candidates.append((area, _order_quad(polygon[:, 0, :])))
                break
    if not candidates:
        raise RuntimeError("Maze frame was not detected. Retake the photo from above.")
    return max(candidates, key=lambda item: item[0])[1]


def _warp(image: np.ndarray, quad: np.ndarray, size: int) -> np.ndarray:
    destination = np.array(
        [[0, 0], [size - 1, 0], [size - 1, size - 1], [0, size - 1]],
        dtype=np.float32,
    )
    matrix = cv2.getPerspectiveTransform(_order_quad(quad), destination)
    return cv2.warpPerspective(image, matrix, (size, size))


def _fit_lattice(projection: np.ndarray, line_count: int) -> tuple[float, float]:
    projection = cv2.GaussianBlur(
        projection.astype(np.float32).reshape(1, -1), (0, 0), 4
    ).ravel()
    projection /= float(projection.max() + 1e-6)
    length = len(projection)
    ideal_step = length / (line_count - 1)
    best = (-math.inf, 0.0, ideal_step)
    for step in np.arange(ideal_step * 0.74, ideal_step * 1.10, 0.5):
        maximum_origin = length - 1 - step * (line_count - 1)
        if maximum_origin < 0:
            continue
        for origin in np.arange(0.0, maximum_origin + 0.25, 0.5):
            score = 0.08 * step * (line_count - 1) / length
            for position in origin + step * np.arange(line_count):
                centre = int(round(position))
                score += float(
                    projection[max(0, centre - 4):min(length, centre + 5)].max()
                )
            if score > best[0]:
                best = (score, float(origin), float(step))
    return best[1], best[2]


def rectify_maze(image: np.ndarray) -> np.ndarray:
    height, width = image.shape[:2]
    scale = min(1.0, 1800.0 / max(height, width))
    small = cv2.resize(image, None, fx=scale, fy=scale) if scale < 1 else image.copy()
    coarse = _warp(small, _maze_quad(small), 1000)
    cyan = _cyan_mask(coarse)
    x_origin, x_step = _fit_lattice(cyan.sum(axis=0), MAZE_CELLS + 1)
    y_origin, y_step = _fit_lattice(cyan.sum(axis=1), MAZE_CELLS + 1)
    grid_quad = np.array(
        [
            [x_origin, y_origin],
            [x_origin + MAZE_CELLS * x_step, y_origin],
            [x_origin + MAZE_CELLS * x_step, y_origin + MAZE_CELLS * y_step],
            [x_origin, y_origin + MAZE_CELLS * y_step],
        ],
        dtype=np.float32,
    )
    return _warp(coarse, grid_quad, MAZE_MM)

## 7. 栅格墙体检测

这一段原本是独立的 `grid_wall_detector.py`。它判断 9x9 迷宫里每一条格子边上到底
有没有墙，做法是多种证据融合，而不是只做一次阈值：

- 局部归一化后的暗色墙体
- 水平/垂直方向的形态学线段
- 墙端头的青色标记
- 沿栅格线的对比度打分

`_compact_object_guard` 会先把小而紧凑的暗色物体（比如圆柱、放在板上的杂物）屏蔽掉，
避免把它们误判成墙。

In [ ]:
from __future__ import annotations

# ---- 7.1 栅格墙体检测（原 grid_wall_detector.py）----
WALL_SCORE = 0.85


def _cyan_mask(image: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (86, 55, 28), (108, 255, 255))
    return cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)),
    )


def _odd(value: float) -> int:
    size = max(3, int(round(value)))
    return size if size % 2 else size + 1


def _component_score(
    mask: np.ndarray,
    vertical: bool,
    segment: int,
    grid_line: int,
    rows: int,
    columns: int,
) -> float:
    height, width = mask.shape
    cell_x, cell_y = width / columns, height / rows
    if vertical:
        y0, y1 = round((segment + 0.08) * cell_y), round((segment + 0.92) * cell_y)
        x0 = max(0, round(grid_line * cell_x - 0.40 * cell_x))
        x1 = min(width, round(grid_line * cell_x + 0.40 * cell_x) + 1)
        along, across, target = cell_y, cell_x, grid_line * cell_x
    else:
        x0, x1 = round((segment + 0.08) * cell_x), round((segment + 0.92) * cell_x)
        y0 = max(0, round(grid_line * cell_y - 0.40 * cell_y))
        y1 = min(height, round(grid_line * cell_y + 0.40 * cell_y) + 1)
        along, across, target = cell_x, cell_y, grid_line * cell_y

    crop = mask[y0:y1, x0:x1]
    count, labels, stats, _ = cv2.connectedComponentsWithStats(crop)
    projection = np.zeros(crop.shape[0] if vertical else crop.shape[1], dtype=bool)
    for label in range(1, count):
        x, y, component_width, component_height, area = stats[label]
        length = component_height if vertical else component_width
        thickness = component_width if vertical else component_height
        if area < cell_x * cell_y * 0.002:
            continue
        if thickness < max(4, round(0.025 * across)):
            continue
        if length < 0.65 * along and length < 1.35 * thickness:
            continue
        axis_edges = (
            (x0 + x, x0 + x + component_width - 1)
            if vertical
            else (y0 + y, y0 + y + component_height - 1)
        )
        if min(abs(value - target) for value in axis_edges) > 0.30 * across:
            continue
        projection |= np.any(labels == label, axis=1 if vertical else 0)

    occupied = np.flatnonzero(projection)
    if not occupied.size:
        return 0.0
    coverage = occupied.size / along
    span = (occupied[-1] - occupied[0] + 1) / along
    if coverage >= 0.65 and span >= 0.65:
        return 1.0
    return 0.70 if coverage >= 0.45 and span >= 0.45 else 0.0


def _score_grid(mask: np.ndarray, vertical: bool, rows: int, columns: int) -> np.ndarray:
    shape = (rows, columns + 1) if vertical else (rows + 1, columns)
    scores = np.zeros(shape, dtype=np.float32)
    for row in range(shape[0]):
        for column in range(shape[1]):
            segment, line = (row, column) if vertical else (column, row)
            scores[row, column] = _component_score(
                mask, vertical, segment, line, rows, columns
            )
    return scores


def _directional_mask(
    gray: np.ndarray,
    cyan_guard: np.ndarray,
    limit: int,
    vertical: bool,
    cell: float,
) -> np.ndarray:
    mask = cv2.inRange(gray, 0, limit)
    mask[cyan_guard > 0] = 0
    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)),
    )
    length = _odd(0.29 * cell)
    kernel = (3, length) if vertical else (length, 3)
    return cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, kernel),
    )


def _local_directional_mask(
    gray: np.ndarray,
    guard: np.ndarray,
    vertical: bool,
    cell: float,
) -> np.ndarray:
    """Extract wall bodies by local contrast when global exposure is uneven."""
    background = cv2.GaussianBlur(gray, (_odd(0.65 * cell), _odd(0.65 * cell)), 0)
    contrast = cv2.subtract(background, gray)
    mask = ((contrast >= 24) & (gray <= 180)).astype(np.uint8) * 255
    mask[guard > 0] = 0
    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)),
    )
    length = _odd(0.27 * cell)
    kernel = (3, length) if vertical else (length, 3)
    return cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, kernel),
    )


def _compact_object_guard(gray: np.ndarray, cell: float, otsu: float) -> np.ndarray:
    """Mask compact dark objects so their silhouettes cannot become grid walls."""
    dark = cv2.inRange(gray, 0, min(105, round(0.78 * otsu)))
    dark = cv2.morphologyEx(
        dark,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    )
    guard = np.zeros_like(gray)
    contours = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, width, height = cv2.boundingRect(contour)
        diameter = max(width, height)
        if not 0.20 * cell <= diameter <= 0.82 * cell:
            continue
        if min(width, height) < 0.55 * diameter or area < 0.020 * cell * cell:
            continue
        (centre_x, centre_y), radius = cv2.minEnclosingCircle(contour)
        if area / max(math.pi * radius * radius, 1.0) < 0.38:
            continue
        cv2.circle(
            guard,
            (round(centre_x), round(centre_y)),
            round(radius + 0.10 * cell),
            255,
            -1,
        )
    return guard


def _line_contrast(
    gray: np.ndarray,
    line: tuple[int, int, int, int],
    horizontal: bool,
    otsu: float,
    cell: float,
) -> float:
    x1, y1, x2, y2 = line
    line_mask = np.zeros_like(gray)
    cv2.line(line_mask, (x1, y1), (x2, y2), 255, max(3, round(0.05 * cell)))
    pixels = gray[line_mask > 0]
    if not pixels.size:
        return 0.0
    line_value = float(np.percentile(pixels, 25))
    padding = max(8, round(0.14 * cell))
    crop = gray[
        max(0, min(y1, y2) - padding):min(gray.shape[0], max(y1, y2) + padding + 1),
        max(0, min(x1, x2) - padding):min(gray.shape[1], max(x1, x2) + padding + 1),
    ]
    contrast = float(np.percentile(crop, 75)) - line_value
    value_limit = min(148.0, otsu + 10.0)
    strict_limit = min(108.0, 0.76 * otsu)
    side_values = []
    for offset in (-max(8, round(0.12 * cell)), max(8, round(0.12 * cell))):
        side = np.zeros_like(gray)
        shift_x, shift_y = (0, offset) if horizontal else (offset, 0)
        cv2.line(
            side,
            (x1 + shift_x, y1 + shift_y),
            (x2 + shift_x, y2 + shift_y),
            255,
            max(3, round(0.05 * cell)),
        )
        values = gray[side > 0]
        side_values.append(float(np.percentile(values, 60)) if values.size else 255.0)
    if min(side_values) < strict_limit + 12.0:
        return 0.0
    if line_value >= value_limit or (
        line_value >= strict_limit and contrast < 0.23 * otsu
    ):
        return 0.0
    darkness = (value_limit - line_value) / max(30.0, 0.40 * otsu)
    contrast_score = contrast / max(35.0, 0.43 * otsu)
    return float(np.clip(max(darkness, contrast_score), 0.0, 1.0))


def _add_thin_lines(
    edges: np.ndarray,
    gray: np.ndarray,
    horizontal_scores: np.ndarray,
    vertical_scores: np.ndarray,
    rows: int,
    columns: int,
    otsu: float,
) -> None:
    cell_x, cell_y = gray.shape[1] / columns, gray.shape[0] / rows
    cell = min(cell_x, cell_y)
    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=max(18, round(0.25 * cell)),
        minLineLength=max(32, round(0.42 * cell)),
        maxLineGap=max(7, round(0.10 * cell)),
    )
    if lines is None:
        return
    h_support = np.zeros_like(horizontal_scores)
    v_support = np.zeros_like(vertical_scores)
    for x1, y1, x2, y2 in lines[:, 0]:
        dx, dy = float(x2 - x1), float(y2 - y1)
        horizontal = abs(dx) >= 0.42 * cell_x and abs(dy) <= max(5.0, 0.12 * abs(dx))
        vertical = abs(dy) >= 0.42 * cell_y and abs(dx) <= max(5.0, 0.12 * abs(dy))
        if not horizontal and not vertical:
            continue
        confidence = _line_contrast(
            gray, (int(x1), int(y1), int(x2), int(y2)), horizontal, otsu, cell
        )
        if confidence < 0.55:
            continue
        if horizontal:
            axis, step, along = 0.5 * (y1 + y2), cell_y, cell_x
            line_count, segment_count = rows, columns
            start, end, support = *sorted((x1, x2)), h_support
        else:
            axis, step, along = 0.5 * (x1 + x2), cell_x, cell_y
            line_count, segment_count = columns, rows
            start, end, support = *sorted((y1, y2)), v_support
        grid_line = round(axis / step)
        if abs(axis - grid_line * step) > 0.18 * step or not 0 <= grid_line <= line_count:
            continue
        for segment in range(segment_count):
            overlap = max(
                0.0,
                min(end, (segment + 1) * along) - max(start, segment * along),
            )
            if overlap < 0.38 * along:
                continue
            index = (grid_line, segment) if horizontal else (segment, grid_line)
            support[index] += overlap * confidence / (0.55 * along)
    np.maximum(horizontal_scores, np.clip(h_support, 0, 1), out=horizontal_scores)
    np.maximum(vertical_scores, np.clip(v_support, 0, 1), out=vertical_scores)


def _add_endpoint_walls(
    edges: np.ndarray,
    gray: np.ndarray,
    cyan: np.ndarray,
    scores: np.ndarray,
    rows: int,
    columns: int,
    otsu: float,
    vertical: bool,
) -> None:
    cell_x, cell_y = gray.shape[1] / columns, gray.shape[0] / rows
    cell = min(cell_x, cell_y)
    radius = max(12, round(0.28 * cell))

    def cyan_pixels(x: int, y: int) -> int:
        return int(
            np.count_nonzero(
                cyan[
                    max(0, y - radius):min(gray.shape[0], y + radius + 1),
                    max(0, x - radius):min(gray.shape[1], x + radius + 1),
                ]
            )
        )

    line_count = columns if vertical else rows
    segment_count = rows if vertical else columns
    for grid_line in range(1, line_count):
        axis = round(grid_line * (cell_x if vertical else cell_y))
        for segment in range(segment_count):
            index = (segment, grid_line) if vertical else (grid_line, segment)
            if scores[index] >= WALL_SCORE:
                continue
            if vertical:
                x0, x1 = max(0, round(axis - 0.24 * cell_x)), min(
                    gray.shape[1], round(axis + 0.24 * cell_x) + 1
                )
                y0, y1 = round(segment * cell_y), round((segment + 1) * cell_y)
                endpoints = ((axis, y0), (axis, y1))
            else:
                x0, x1 = round(segment * cell_x), round((segment + 1) * cell_x)
                y0, y1 = max(0, round(axis - 0.24 * cell_y)), min(
                    gray.shape[0], round(axis + 0.24 * cell_y) + 1
                )
                endpoints = ((x0, axis), (x1, axis))
            if min(cyan_pixels(*point) for point in endpoints) < 0.009 * cell * cell:
                continue
            lines = cv2.HoughLinesP(
                edges[y0:y1, x0:x1],
                1,
                np.pi / 180,
                threshold=max(10, round(0.12 * cell)),
                minLineLength=max(24, round(0.30 * (cell_y if vertical else cell_x))),
                maxLineGap=max(10, round(0.18 * (cell_y if vertical else cell_x))),
            )
            for x_start, y_start, x_end, y_end in (
                [] if lines is None else lines[:, 0]
            ):
                dx, dy = float(x_end - x_start), float(y_end - y_start)
                length = abs(dy) if vertical else abs(dx)
                cross = abs(dx) if vertical else abs(dy)
                along = cell_y if vertical else cell_x
                if length < 0.50 * along or cross > max(6.0, 0.12 * length):
                    continue
                line = (
                    int(x_start + x0),
                    int(y_start + y0),
                    int(x_end + x0),
                    int(y_end + y0),
                )
                measured_axis = 0.5 * (
                    (line[0] + line[2]) if vertical else (line[1] + line[3])
                )
                if abs(measured_axis - axis) > 0.18 * (cell_x if vertical else cell_y):
                    continue
                if _line_contrast(gray, line, not vertical, otsu, cell) >= 0.55:
                    scores[index] = 1.0
                    break


def _promote_parallel(scores: np.ndarray, candidates: np.ndarray, axis: int) -> None:
    strong = scores >= 0.85
    neighbour = np.zeros_like(strong)
    if axis == 1:
        neighbour[:, 1:] |= strong[:, :-1]
        neighbour[:, :-1] |= strong[:, 1:]
    else:
        neighbour[1:] |= strong[:-1]
        neighbour[:-1] |= strong[1:]
    scores[(candidates >= 0.65) & ~strong & neighbour] = 1.0


def _promote_relaxed_vertical_corners(
    vertical: np.ndarray,
    relaxed: np.ndarray,
    horizontal: np.ndarray,
) -> None:
    rows, line_count = vertical.shape
    columns = line_count - 1
    for row in range(rows):
        for column in range(line_count):
            if vertical[row, column] >= 0.85 or relaxed[row, column] < 0.85:
                continue
            top = (
                (column > 0 and horizontal[row, column - 1] >= 0.85)
                or (column < columns and horizontal[row, column] >= 0.85)
            )
            bottom = (
                (column > 0 and horizontal[row + 1, column - 1] >= 0.85)
                or (column < columns and horizontal[row + 1, column] >= 0.85)
            )
            if top and bottom:
                vertical[row, column] = 1.0


def detect_grid_walls(
    image: np.ndarray,
    rows: int = 9,
    columns: int = 9,
    ignored_circles: tuple[tuple[float, float, float], ...] = (),
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    otsu, _ = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cyan = _cyan_mask(image)
    cell_x, cell_y = image.shape[1] / columns, image.shape[0] / rows
    cell = min(cell_x, cell_y)
    guard = cv2.bitwise_or(
        _compact_object_guard(blurred, cell, float(otsu)),
        cv2.dilate(
        cyan, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        ),
    )
    for x, y, radius in ignored_circles:
        cv2.circle(guard, (round(x), round(y)), round(radius), 255, -1)
    horizontal_mask = _directional_mask(
        blurred, guard, min(108, round(0.76 * otsu)), False, cell_x
    )
    vertical_mask = _directional_mask(
        blurred, guard, min(108, round(0.76 * otsu)), True, cell_y
    )
    relaxed_vertical = _directional_mask(
        blurred, guard, min(160, round(1.08 * otsu)), True, cell_y
    )
    horizontal = _score_grid(horizontal_mask, False, rows, columns)
    vertical = _score_grid(vertical_mask, True, rows, columns)
    relaxed_scores = _score_grid(relaxed_vertical, True, rows, columns)
    local_horizontal = _score_grid(
        _local_directional_mask(blurred, guard, False, cell_x),
        False,
        rows,
        columns,
    )
    local_vertical = _score_grid(
        _local_directional_mask(blurred, guard, True, cell_y),
        True,
        rows,
        columns,
    )
    np.maximum(horizontal, local_horizontal, out=horizontal)
    np.maximum(vertical, local_vertical, out=vertical)

    enhanced = cv2.createCLAHE(clipLimit=1.8, tileGridSize=(8, 8)).apply(gray)
    edges = cv2.Canny(
        cv2.GaussianBlur(enhanced, (3, 3), 0),
        max(35, round(0.36 * otsu)),
        max(100, round(1.02 * otsu)),
        L2gradient=True,
    )
    edges[cv2.dilate(guard, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))) > 0] = 0
    _add_thin_lines(edges, gray, horizontal, vertical, rows, columns, float(otsu))
    _add_endpoint_walls(edges, gray, cyan, horizontal, rows, columns, float(otsu), False)
    _add_endpoint_walls(edges, gray, cyan, vertical, rows, columns, float(otsu), True)
    _promote_parallel(horizontal, horizontal, 1)
    _promote_parallel(vertical, vertical, 0)
    _promote_parallel(vertical, relaxed_scores, 0)
    _promote_relaxed_vertical_corners(vertical, relaxed_scores, horizontal)

    # A 0.70 component has a continuous wall body over almost half a cell.
    # Compact dark objects have already been removed, so this is conservative
    # for walls while rejecting isolated floor marks and connector pieces.
    horizontal[horizontal >= 0.65] = 1.0
    vertical[vertical >= 0.65] = 1.0

    horizontal_walls = horizontal >= WALL_SCORE
    vertical_walls = vertical >= WALL_SCORE
    horizontal_walls[0, :] = horizontal_walls[-1, :] = True
    vertical_walls[:, 0] = vertical_walls[:, -1] = True
    return horizontal_walls, vertical_walls, horizontal, vertical

## 8. 全局地图：墙 + 圆柱 -> 占据栅格

`extract_global_map` 把矫正图变成规划用的地图：检测圆柱、检测墙、然后按安全边界
膨胀成占据栅格。如果画面被人或者其他东西挡住太多，这里会直接报错要求重拍，
而不是猜一个墙出来。

In [ ]:
from __future__ import annotations

# ---- 8.1 圆柱检测、全局占据地图、栅格迷宫 ----
def _detect_cylinders(dark: np.ndarray, config: PlannerConfig) -> list[tuple[float, float]]:
    """Find fixed-size cylinders even when their silhouettes touch a wall.

    When the course position is still unknown the whole board is searched,
    because locating the course needs the cylinders. That is safe: the filter
    below demands a dark blob at least 70 mm across in both directions, and a
    12 mm maze wall never produces one. The chamfered corners are the one
    exception, so they are excluded explicitly.
    """
    distance = cv2.distanceTransform(dark, cv2.DIST_L2, 5)
    core = np.zeros_like(dark)
    inset = round(config.wall_safety_mm)
    known = config.course_top_row is not None and config.course_left_column is not None
    if known:
        top = config.course_top_row * CELL_MM + inset
        left = config.course_left_column * CELL_MM + inset
        bottom = (config.course_top_row + COURSE_CELLS) * CELL_MM - inset
        right = (config.course_left_column + COURSE_CELLS) * CELL_MM - inset
    else:
        top = left = inset
        bottom = right = MAZE_MM - inset
    minimum_radius = 0.30 * config.obstacle_diameter_mm
    core[top:bottom, left:right] = (distance[top:bottom, left:right] >= minimum_radius) * 255
    if not known:
        # The cut-off corner cells are large dark areas and would otherwise
        # register as cylinders.
        for row, column in BLOCKED_CORNERS:
            core[row * CELL_MM:(row + 1) * CELL_MM,
                 column * CELL_MM:(column + 1) * CELL_MM] = 0

    obstacles: list[tuple[float, float]] = []
    count, labels, stats, _ = cv2.connectedComponentsWithStats(core)
    for label in range(1, count):
        if stats[label, cv2.CC_STAT_AREA] < 200:
            continue
        ys, xs = np.where(labels == label)
        peak = int(np.argmax(distance[ys, xs]))
        radius = distance[ys[peak], xs[peak]]
        if 0.35 * config.obstacle_diameter_mm <= radius <= 0.70 * config.obstacle_diameter_mm:
            obstacles.append((float(xs[peak]), float(ys[peak])))
    return sorted(obstacles, key=lambda point: (point[1], point[0]))


def extract_global_map(rectified: np.ndarray, config: PlannerConfig) -> GlobalMap:
    hsv = cv2.cvtColor(rectified, cv2.COLOR_BGR2HSV)
    gray_raw = cv2.cvtColor(rectified, cv2.COLOR_BGR2GRAY)
    dark_fraction = float(np.mean(gray_raw < 80))
    coloured_fraction = float(np.mean(hsv[:, :, 1] > 70))
    if dark_fraction > 0.28 and coloured_fraction > 0.12:
        raise ValueError(
            "Maze is heavily occluded; take another overhead photo before planning"
        )
    gray = cv2.GaussianBlur(gray_raw, (7, 7), 0)
    otsu, _ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    dark_threshold = int(np.clip(0.80 * otsu, 65, 105))
    dark = (gray < dark_threshold).astype(np.uint8) * 255
    dark = cv2.morphologyEx(
        dark,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    )

    obstacles = _detect_cylinders(dark, config)
    wall_source = dark.copy()
    obstacle_guard = round(config.obstacle_diameter_mm / 2 + 10)
    for x, y in obstacles:
        cv2.circle(wall_source, (round(x), round(y)), obstacle_guard, 0, -1)

    wall_mask = np.zeros_like(dark)
    contours = cv2.findContours(wall_source, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    for contour in contours:
        area = cv2.contourArea(contour)
        _, _, width, height = cv2.boundingRect(contour)
        diameter = max(width, height)
        if area >= 280 and (diameter >= 65 or area >= 1400):
            cv2.drawContours(wall_mask, [contour], -1, 255, -1)

    # Recover reflective wall bodies that are lighter than the contour threshold.
    wall_body = cv2.inRange(gray, 0, min(130, dark_threshold + 30))
    for x, y in obstacles:
        cv2.circle(wall_body, (round(x), round(y)), obstacle_guard, 0, -1)
    wall_body = cv2.morphologyEx(
        wall_body,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7)),
    )
    horizontal = cv2.morphologyEx(
        wall_body,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (55, 9)),
    )
    vertical = cv2.morphologyEx(
        wall_body,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (9, 55)),
    )
    wall_mask = cv2.bitwise_or(
        wall_mask,
        cv2.bitwise_or(horizontal, vertical),
    )

    for row, column in BLOCKED_CORNERS:
        cv2.rectangle(
            wall_mask,
            (column * CELL_MM, row * CELL_MM),
            ((column + 1) * CELL_MM - 1, (row + 1) * CELL_MM - 1),
            255,
            -1,
        )
    cv2.rectangle(wall_mask, (0, 0), (MAZE_MM - 1, MAZE_MM - 1), 255, 3)

    horizontal_walls, vertical_walls, horizontal_scores, vertical_scores = (
        detect_grid_walls(
            rectified,
            MAZE_CELLS,
            MAZE_CELLS,
            tuple(
                (x, y, 0.42 * CELL_MM)
                for x, y in obstacles
            ),
        )
    )

    # The course position is decided here, from the walls and the cylinders,
    # because everything downstream needs it. Cylinders found outside the
    # course are dropped: on this board a cylinder only exists inside it.
    partial = GlobalMap(
        rectified, wall_mask, wall_mask, obstacles, dark_threshold,
        horizontal_walls, vertical_walls, horizontal_scores, vertical_scores,
    )
    if config.course_top_row is not None and config.course_left_column is not None:
        top_row = config.course_top_row
        left_column = config.course_left_column
        candidates: tuple = ()
    else:
        ranked = rank_course_positions(partial, limit=5)
        best = detect_course_position(partial)
        top_row, left_column = best.top_row, best.left_column
        candidates = tuple(ranked)

    obstacles = [
        (x, y)
        for x, y in obstacles
        if left_column * CELL_MM <= x < (left_column + COURSE_CELLS) * CELL_MM
        and top_row * CELL_MM <= y < (top_row + COURSE_CELLS) * CELL_MM
    ]

    wall_radius = int(round(config.wall_clearance_mm))
    occupancy = cv2.dilate(
        wall_mask,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * wall_radius + 1, 2 * wall_radius + 1),
        ),
    )
    cylinder_radius = int(round(config.cylinder_clearance_mm))
    for x, y in obstacles:
        cv2.circle(occupancy, (round(x), round(y)), cylinder_radius, 255, -1)

    return GlobalMap(
        rectified,
        wall_mask,
        occupancy,
        obstacles,
        dark_threshold,
        horizontal_walls,
        vertical_walls,
        horizontal_scores,
        vertical_scores,
        top_row,
        left_column,
        candidates,
    )


def _edge_coverage(
    wall_mask: np.ndarray,
    horizontal: bool,
    grid_line: int,
    segment: int,
) -> float:
    line = grid_line * CELL_MM
    start = segment * CELL_MM + 25
    end = (segment + 1) * CELL_MM - 25
    half_width = 50
    if horizontal:
        region = wall_mask[
            max(0, line - half_width):min(MAZE_MM, line + half_width + 1),
            start:end,
        ]
        return float(np.mean(np.any(region > 0, axis=0)))
    region = wall_mask[
        start:end,
        max(0, line - half_width):min(MAZE_MM, line + half_width + 1),
    ]
    return float(np.mean(np.any(region > 0, axis=1)))


def _grid_coverage(wall_mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Wall pixel coverage of every grid segment in the board.

    Returned as horizontal[line_row, segment_column] and
    vertical[line_column, segment_row], both in 0..1. Measuring coverage once
    for all 180 segments keeps the course search to pure array indexing.
    """
    horizontal = np.zeros((MAZE_CELLS + 1, MAZE_CELLS), dtype=np.float32)
    vertical = np.zeros((MAZE_CELLS + 1, MAZE_CELLS), dtype=np.float32)
    for line in range(MAZE_CELLS + 1):
        for segment in range(MAZE_CELLS):
            horizontal[line, segment] = _edge_coverage(wall_mask, True, line, segment)
            vertical[line, segment] = _edge_coverage(wall_mask, False, line, segment)
    return horizontal, vertical


@dataclass(frozen=True)
class CourseCandidate:
    top_row: int
    left_column: int
    score: float
    interior_wall: float      # 0 is a clean open area, 1 is a normal maze block
    boundary_wall: float      # 1 is fully enclosed, lower means more openings
    cylinders_inside: int
    cylinders_total: int

    @property
    def position(self) -> tuple[int, int]:
        return (self.top_row, self.left_column)


def _course_overlaps_blocked_corner(top: int, left: int) -> bool:
    return any(
        top <= row < top + COURSE_CELLS and left <= column < left + COURSE_CELLS
        for row, column in BLOCKED_CORNERS
    )


def rank_course_positions(
    map_data: GlobalMap,
    limit: int = 5,
) -> list[CourseCandidate]:
    """Score every legal 5 x 5 position and return the best ones first.

    Three independent pieces of evidence separate the obstacle course from an
    ordinary block of the maze:

    1. Its interior is empty. A normal 5 x 5 block of a 9 x 9 maze contains
       many of the 40 internal wall segments; the course contains none. This
       is by far the strongest signal.
    2. Its boundary is a closed ring apart from a few portals, so most of the
       20 perimeter segments carry wall.
    3. Every detected cylinder lies inside it, and none lies outside.

    Coverage of the wall mask is used rather than the thresholded wall
    decisions, so a wall that was too faint to be classified still counts.
    """
    horizontal, vertical = _grid_coverage(map_data.wall_mask)
    obstacles = map_data.obstacles_mm
    span = MAZE_CELLS - COURSE_CELLS

    candidates: list[CourseCandidate] = []
    for top in range(span + 1):
        for left in range(span + 1):
            if _course_overlaps_blocked_corner(top, left):
                continue

            # Interior: 4 internal lines each way, 5 segments per line.
            interior = float(np.mean([
                float(np.mean(horizontal[top + 1:top + COURSE_CELLS,
                                         left:left + COURSE_CELLS])),
                float(np.mean(vertical[left + 1:left + COURSE_CELLS,
                                       top:top + COURSE_CELLS])),
            ]))

            # Perimeter: the four sides of the ring.
            boundary = float(np.mean(np.concatenate([
                horizontal[top, left:left + COURSE_CELLS],
                horizontal[top + COURSE_CELLS, left:left + COURSE_CELLS],
                vertical[left, top:top + COURSE_CELLS],
                vertical[left + COURSE_CELLS, top:top + COURSE_CELLS],
            ])))

            inside = sum(
                1
                for x, y in obstacles
                if left * CELL_MM <= x < (left + COURSE_CELLS) * CELL_MM
                and top * CELL_MM <= y < (top + COURSE_CELLS) * CELL_MM
            )
            if obstacles:
                cylinder_term = inside / len(obstacles)
            else:
                cylinder_term = 0.5      # no evidence either way

            score = (
                2.5 * (1.0 - interior)
                + 1.0 * boundary
                + 1.5 * cylinder_term
            ) / 5.0
            candidates.append(CourseCandidate(
                top, left, score, interior, boundary, inside, len(obstacles),
            ))

    candidates.sort(key=lambda item: item.score, reverse=True)
    return candidates[:limit] if limit else candidates


# A real course leaves almost nothing inside and wins by a clear margin.
# Measured over the supplied photographs: boards holding a course score an
# interior of 0.005 to 0.06 and beat the runner up by 0.11 to 0.22, while
# plain maze boards sit at 0.13 to 0.30 interior and 0.005 to 0.048 margin.
COURSE_MAX_INTERIOR = 0.22
COURSE_MIN_MARGIN = 0.05


def detect_course_position(map_data: GlobalMap) -> CourseCandidate:
    ranked = rank_course_positions(map_data, limit=5)
    if not ranked:
        raise RuntimeError("No legal 5 x 5 course position exists on this board")
    best = ranked[0]
    margin = best.score - ranked[1].score if len(ranked) > 1 else 1.0

    if best.interior_wall > COURSE_MAX_INTERIOR or margin < COURSE_MIN_MARGIN:
        table = "\n".join(
            f"    {c.position}  score {c.score:.3f}  interior {c.interior_wall:.3f}"
            f"  boundary {c.boundary_wall:.3f}  cylinders {c.cylinders_inside}"
            for c in ranked
        )
        raise RuntimeError(
            "No 5 x 5 obstacle course was recognised in this photograph.\n"
            f"  best guess {best.position}, interior wall {best.interior_wall:.3f} "
            f"(want below {COURSE_MAX_INTERIOR}), margin {margin:.3f} "
            f"(want above {COURSE_MIN_MARGIN}).\n"
            "  A course interior is empty; every candidate here still has walls\n"
            "  in it, so this is most likely a plain maze photograph.\n"
            "  Ranked candidates:\n" + table + "\n"
            "  If the course really is there, set course_top_row and\n"
            "  course_left_column by hand to override the search."
        )
    return best


def resolve_course_config(
    map_data: GlobalMap,
    config: PlannerConfig,
) -> PlannerConfig:
    """Fills in the course position when it was left as None.

    extract_global_map already decided this, so no image work is repeated.
    """
    if config.course_top_row is not None and config.course_left_column is not None:
        return config
    return replace(
        config,
        course_top_row=map_data.course_top_row,
        course_left_column=map_data.course_left_column,
    )


def detect_course_portals(
    map_data: GlobalMap,
    config: PlannerConfig,
) -> tuple[CoursePortal, ...]:
    config = resolve_course_config(map_data, config)
    top, left = config.course_top_row, config.course_left_column
    edges = (
        ("N", True, top, left, map_data.horizontal_walls[top, left:left + 5]),
        ("E", False, left + 5, top, map_data.vertical_walls[top:top + 5, left + 5]),
        ("S", True, top + 5, left, map_data.horizontal_walls[top + 5, left:left + 5]),
        ("W", False, left, top, map_data.vertical_walls[top:top + 5, left]),
    )
    portals: list[CoursePortal] = []
    for side, horizontal, line, first_segment, walls in edges:
        for offset in range(5):
            coverage = _edge_coverage(
                map_data.wall_mask,
                horizontal,
                line,
                first_segment + offset,
            )
            if not walls[offset] and coverage < config.portal_open_threshold:
                portals.append(CoursePortal(side, offset))
    return tuple(portals)


def extract_grid_maze(map_data: GlobalMap, config: PlannerConfig) -> GridMaze:
    config = resolve_course_config(map_data, config)
    horizontal = map_data.horizontal_walls.copy()
    vertical = map_data.vertical_walls.copy()

    blocked = np.zeros((MAZE_CELLS, MAZE_CELLS), dtype=bool)
    for row, column in BLOCKED_CORNERS:
        blocked[row, column] = True
    top, left = config.course_top_row, config.course_left_column
    blocked[top:top + 5, left:left + 5] = True
    return GridMaze(horizontal, vertical, blocked)

## 9. 门洞识别与栅格路径规划

`detect_course_portals` 沿 5x5 场地的 20 段边界逐段测量，自动找出哪些位置是开口。
开口的数量和位置每张图都可能不一样，代码不做任何假设。

`plan_grid_commands` 是带朝向的栅格搜索（状态是 `(行, 列, 朝向)`，转弯有代价），
输出 `f`/`l`/`r` 指令串。5x5 区域内的格子在这里是被封死的，所以纯栅格路线不会
误穿场地。

In [ ]:
from __future__ import annotations

# ---- 9.1 门洞位姿与栅格规划 ----
def _portal_pose(portal: CoursePortal, config: PlannerConfig) -> PortalPose:
    if portal.side not in {"N", "E", "S", "W"} or not 0 <= portal.offset < 5:
        raise ValueError(f"Invalid course portal: {portal}")
    top, left = config.course_top_row, config.course_left_column
    if portal.side == "N":
        inside = (top, left + portal.offset)
        outside = (top - 1, left + portal.offset)
        inward = 2
    elif portal.side == "E":
        inside = (top + portal.offset, left + 4)
        outside = (top + portal.offset, left + 5)
        inward = 3
    elif portal.side == "S":
        inside = (top + 4, left + portal.offset)
        outside = (top + 5, left + portal.offset)
        inward = 0
    else:
        inside = (top + portal.offset, left)
        outside = (top + portal.offset, left - 1)
        inward = 1
    if not all(0 <= value < MAZE_CELLS for cell in (inside, outside) for value in cell):
        raise ValueError(f"Course portal lies outside the maze: {portal}")
    return PortalPose(portal, inside, outside, inward)


def _cell_centre(cell: tuple[int, int]) -> tuple[float, float]:
    return (
        (cell[1] + 0.5) * CELL_MM,
        (cell[0] + 0.5) * CELL_MM,
    )


def plan_grid_commands(
    maze: GridMaze,
    start: tuple[int, int, int],
    goal: tuple[int, int, int],
) -> tuple[str, list[tuple[int, int]]]:
    if maze.blocked_cells[start[0], start[1]] or maze.blocked_cells[goal[0], goal[1]]:
        return "", []
    queue = [(0, 0, 0, start)]
    costs = {start: (0, 0)}
    previous: dict[tuple[int, int, int], tuple[tuple[int, int, int], str]] = {}
    sequence = 0
    while queue:
        forwards, turns, _, state = heapq.heappop(queue)
        if costs.get(state) != (forwards, turns):
            continue
        if state == goal:
            break
        row, column, direction = state
        transitions = [
            ("l", (row, column, (direction - 1) % 4), (forwards, turns + 1)),
            ("r", (row, column, (direction + 1) % 4), (forwards, turns + 1)),
        ]
        if maze.can_move(row, column, direction):
            delta_row, delta_column = DELTAS[direction]
            transitions.append(
                (
                    "f",
                    (row + delta_row, column + delta_column, direction),
                    (forwards + 1, turns),
                )
            )
        for command, next_state, next_cost in transitions:
            if next_cost >= costs.get(next_state, (math.inf, math.inf)):
                continue
            costs[next_state] = next_cost
            previous[next_state] = (state, command)
            sequence += 1
            heapq.heappush(queue, (*next_cost, sequence, next_state))
    if goal not in costs:
        return "", []

    commands: list[str] = []
    state = goal
    while state != start:
        state, command = previous[state]
        commands.append(command)
    commands.reverse()
    row, column, direction = start
    cells = [(row, column)]
    for command in commands:
        if command == "l":
            direction = (direction - 1) % 4
        elif command == "r":
            direction = (direction + 1) % 4
        else:
            delta_row, delta_column = DELTAS[direction]
            row += delta_row
            column += delta_column
            cells.append((row, column))
    return "".join(commands), cells


def _course_map(map_data: GlobalMap, config: PlannerConfig) -> GlobalMap:
    top = config.course_top_row * CELL_MM
    left = config.course_left_column * CELL_MM
    bottom = top + COURSE_CELLS * CELL_MM
    right = left + COURSE_CELLS * CELL_MM
    occupancy = np.full_like(map_data.occupancy_mask, 255)
    occupancy[top:bottom, left:right] = map_data.occupancy_mask[top:bottom, left:right]

    half_width = int(round(config.wall_clearance_mm))
    depth = int(round(0.65 * CELL_MM))
    for portal in detect_course_portals(map_data, config):
        pose = _portal_pose(portal, config)
        centre_x, centre_y = (round(value) for value in _cell_centre(pose.inside_cell))
        if portal.side == "N":
            corners = (centre_x - half_width, top, centre_x + half_width, top + depth)
        elif portal.side == "S":
            corners = (centre_x - half_width, bottom - depth, centre_x + half_width, bottom)
        elif portal.side == "W":
            corners = (left, centre_y - half_width, left + depth, centre_y + half_width)
        else:
            corners = (right - depth, centre_y - half_width, right, centre_y + half_width)
        cv2.rectangle(occupancy, corners[:2], corners[2:], 0, -1)

    # Portal carving removes boundary-wall inflation only; cylinders stay protected.
    cylinder_radius = int(round(config.cylinder_clearance_mm))
    for x, y in map_data.obstacles_mm:
        cv2.circle(occupancy, (round(x), round(y)), cylinder_radius, 255, -1)
    return GlobalMap(
        map_data.rectified,
        map_data.wall_mask,
        occupancy,
        map_data.obstacles_mm,
        map_data.dark_threshold,
        map_data.horizontal_walls,
        map_data.vertical_walls,
        map_data.horizontal_scores,
        map_data.vertical_scores,
    )

## 10. 5x5 内部的连续路径规划

场地内部不再按格子走，而是走真正的连续折线：

1. `_astar`：10 mm 分辨率的 A*，先找出一条安全走廊；
2. `_smooth_path`：整条路径做视线简化，去掉多余的拐点；
3. `_quantise_path`：把每一段的朝向量化成**整数度**（Arduino 只接受整数），
   并且用小范围搜索保证终点误差不超过 10 mm；
4. `_segment_clear`：量化之后再验一遍，确认没有蹭到膨胀区。

In [ ]:
from __future__ import annotations

# ---- 10.1 连续 A*、平滑、整数航向量化 ----
def _normalise_angle(angle: int | float) -> int:
    return int((round(angle) + 180) % 360 - 180)


def _map_heading(start: tuple[float, float], end: tuple[float, float]) -> int:
    dx, dy = end[0] - start[0], end[1] - start[1]
    return _normalise_angle(math.degrees(math.atan2(-dx, -dy)))


def _endpoint(start: tuple[float, float], heading: int, distance: int) -> tuple[float, float]:
    radians = math.radians(heading)
    return start[0] - distance * math.sin(radians), start[1] - distance * math.cos(radians)


def _point_clear(point: tuple[float, float], occupancy: np.ndarray) -> bool:
    x, y = point
    if not (0 <= x < MAZE_MM and 0 <= y < MAZE_MM):
        return False
    column = int(np.clip(round(x), 0, occupancy.shape[1] - 1))
    row = int(np.clip(round(y), 0, occupancy.shape[0] - 1))
    return occupancy[row, column] == 0


def _segment_clear(
    start: tuple[float, float],
    end: tuple[float, float],
    occupancy: np.ndarray,
) -> bool:
    if not _point_clear(start, occupancy) or not _point_clear(end, occupancy):
        return False
    distance = math.hypot(end[0] - start[0], end[1] - start[1])
    count = max(2, int(math.ceil(distance / 3.0)) + 1)
    x = np.clip(
        np.rint(np.linspace(start[0], end[0], count)),
        0,
        occupancy.shape[1] - 1,
    ).astype(np.int32)
    y = np.clip(
        np.rint(np.linspace(start[1], end[1], count)),
        0,
        occupancy.shape[0] - 1,
    ).astype(np.int32)
    return not np.any(occupancy[y, x])


def _nearest_node(
    point: tuple[float, float],
    occupancy: np.ndarray,
    step: int,
) -> tuple[int, int]:
    maximum = (MAZE_MM - 1) // step
    target = (
        int(np.clip(round(point[0] / step), 0, maximum)),
        int(np.clip(round(point[1] / step), 0, maximum)),
    )
    for radius in range(13):
        ring = [
            (target[0] + dx, target[1] + dy)
            for dx in range(-radius, radius + 1)
            for dy in range(-radius, radius + 1)
            if max(abs(dx), abs(dy)) == radius
        ]
        ring.sort(key=lambda node: math.hypot(node[0] - target[0], node[1] - target[1]))
        for node in ring:
            if not (0 <= node[0] <= maximum and 0 <= node[1] <= maximum):
                continue
            if _point_clear((node[0] * step, node[1] * step), occupancy):
                return node
    raise ValueError(f"Point {point} is not connected to planning grid")


def _astar(
    start_mm: tuple[float, float],
    goal_mm: tuple[float, float],
    occupancy: np.ndarray,
    config: PlannerConfig,
) -> list[tuple[float, float]]:
    step = config.grid_mm
    start = _nearest_node(start_mm, occupancy, step)
    goal = _nearest_node(goal_mm, occupancy, step)
    moves = (
        (-1, -1), (0, -1), (1, -1),
        (-1, 0),             (1, 0),
        (-1, 1),  (0, 1),   (1, 1),
    )
    maximum = (MAZE_MM - 1) // step
    queue: list[tuple[float, float, tuple[int, int]]] = [(0.0, 0.0, start)]
    cost = {start: 0.0}
    parent: dict[tuple[int, int], tuple[int, int]] = {}
    visited: set[tuple[int, int]] = set()

    while queue:
        _, current_cost, current = heapq.heappop(queue)
        if current in visited:
            continue
        visited.add(current)
        if current == goal:
            break
        current_mm = (current[0] * step, current[1] * step)
        for dx, dy in moves:
            neighbour = (current[0] + dx, current[1] + dy)
            if not (0 <= neighbour[0] <= maximum and 0 <= neighbour[1] <= maximum):
                continue
            neighbour_mm = (neighbour[0] * step, neighbour[1] * step)
            if not _segment_clear(current_mm, neighbour_mm, occupancy):
                continue
            new_cost = current_cost + step * math.hypot(dx, dy)
            if new_cost >= cost.get(neighbour, math.inf):
                continue
            cost[neighbour] = new_cost
            parent[neighbour] = current
            heuristic = step * math.hypot(goal[0] - neighbour[0], goal[1] - neighbour[1])
            heapq.heappush(queue, (new_cost + heuristic, new_cost, neighbour))

    if goal not in cost:
        raise RuntimeError("No collision-free global route exists between these poses")
    nodes = [goal]
    while nodes[-1] != start:
        nodes.append(parent[nodes[-1]])
    nodes.reverse()
    points = [(node[0] * step, node[1] * step) for node in nodes]
    points[0] = start_mm
    points[-1] = goal_mm
    return points


def _smooth_path(
    path: Sequence[tuple[float, float]],
    occupancy: np.ndarray,
    minimum_distance: float,
) -> list[tuple[float, float]]:
    costs: list[tuple[int, float] | None] = [None] * len(path)
    previous = [-1] * len(path)
    costs[0] = (0, 0.0)
    for end in range(1, len(path)):
        for start in range(end):
            if costs[start] is None:
                continue
            distance = math.dist(path[start], path[end])
            if distance < minimum_distance:
                continue
            if not _segment_clear(path[start], path[end], occupancy):
                continue
            candidate = (costs[start][0] + 1, costs[start][1] + distance)
            if costs[end] is None or candidate < costs[end]:
                costs[end] = candidate
                previous[end] = start
    if costs[-1] is None:
        raise RuntimeError("No executable straight-segment route exists")
    indices = [len(path) - 1]
    while indices[-1] != 0:
        indices.append(previous[indices[-1]])
    return [path[index] for index in reversed(indices)]


def _split_long_segments(
    points: Sequence[tuple[float, float]],
    maximum_distance: int,
) -> list[tuple[float, float]]:
    split = [points[0]]
    for start, end in zip(points, points[1:]):
        distance = math.hypot(end[0] - start[0], end[1] - start[1])
        parts = max(1, int(math.ceil(distance / maximum_distance)))
        for part in range(1, parts + 1):
            fraction = part / parts
            split.append(
                (
                    start[0] + fraction * (end[0] - start[0]),
                    start[1] + fraction * (end[1] - start[1]),
                )
            )
    return split


def _quantise_path(
    points: Sequence[tuple[float, float]],
    occupancy: np.ndarray,
    config: PlannerConfig,
) -> tuple[list[tuple[float, float]], list[Motion]]:
    targets = _split_long_segments(points, config.maximum_segment_mm)
    states = [(0.0, targets[0], [], [], [targets[0]])]
    for index, target in enumerate(targets[1:], 1):
        candidates = []
        for score, actual, headings, distances, actual_points in states:
            ideal_heading = _map_heading(actual, target)
            ideal_distance = max(1, int(round(math.dist(actual, target))))
            for heading_offset in sorted(range(-8, 9), key=abs):
                heading = _normalise_angle(ideal_heading + heading_offset)
                for distance_offset in sorted(range(-10, 11), key=abs):
                    distance = max(1, ideal_distance + distance_offset)
                    end = _endpoint(actual, heading, distance)
                    if not _segment_clear(actual, end, occupancy):
                        continue
                    error = math.dist(end, target)
                    candidates.append(
                        (
                            score + error + 0.05 * abs(heading_offset),
                            end,
                            headings + [heading],
                            distances + [distance],
                            actual_points + [end],
                        )
                    )
        if not candidates:
            raise RuntimeError("Whole-degree rounding removed the required clearance")
        final_segment = index == len(targets) - 1
        candidates.sort(
            key=lambda state: (
                math.dist(state[1], target) if final_segment else state[0],
                state[0],
            )
        )
        states = []
        seen = set()
        for state in candidates:
            key = (round(state[1][0]), round(state[1][1]), state[2][-1])
            if key in seen:
                continue
            seen.add(key)
            states.append(state)
            if len(states) == 40:
                break

    _, _, headings, distances, actual_points = states[0]

    motions: list[Motion] = []
    previous_heading = _normalise_angle(config.start_heading_deg)
    for heading, distance in zip(headings, distances):
        motions.append(Motion(_normalise_angle(heading - previous_heading), distance, heading))
        previous_heading = heading
    if config.goal_heading_deg is not None:
        final_heading = _normalise_angle(config.goal_heading_deg)
        final_turn = _normalise_angle(final_heading - previous_heading)
        if final_turn:
            motions.append(Motion(final_turn, 0, final_heading))
    return actual_points, motions


def _plan_on_map(
    map_data: GlobalMap,
    start_mm: tuple[float, float],
    goal_mm: tuple[float, float],
    config: PlannerConfig,
    render: bool = True,
) -> PlanResult:
    if not _point_clear(start_mm, map_data.occupancy_mask):
        raise ValueError("Start pose is inside a wall or safety area")
    if not _point_clear(goal_mm, map_data.occupancy_mask):
        raise ValueError("Goal pose is inside a wall or safety area")
    grid_path = _astar(start_mm, goal_mm, map_data.occupancy_mask, config)
    path = _smooth_path(
        grid_path,
        map_data.occupancy_mask,
        config.minimum_course_segment_mm,
    )
    waypoints, motions = _quantise_path(path, map_data.occupancy_mask, config)
    if render:
        occupancy_image = draw_occupancy(map_data, config)
        route_image = draw_route(map_data, waypoints, motions, config)
    else:
        occupancy_image = np.empty((0, 0, 3), dtype=np.uint8)
        route_image = np.empty((0, 0, 3), dtype=np.uint8)
    result = PlanResult(
        map_data,
        config,
        [start_mm, goal_mm],
        waypoints,
        motions,
        occupancy_image,
        route_image,
        _normalise_angle(config.start_heading_deg),
        None if config.goal_heading_deg is None else _normalise_angle(config.goal_heading_deg),
    )
    validate_result(result)
    return result

### 10.1 混合规划：比较所有可能的进出口组合

`plan_hybrid_on_map` 会把两类方案放在一起比：

- **纯栅格**：完全不进 5x5 场地；
- **穿场地**：对每一对（进口, 出口）门洞组合都算一遍
  `栅格 -> 5x5 连续 -> 栅格`，取最短的。

穿过门洞时只解除门洞处的边界膨胀，圆柱的安全区始终保留。

In [ ]:
from __future__ import annotations

# ---- 10.2 混合规划入口 ----
def plan_hybrid_on_map(
    map_data: GlobalMap,
    start_pose: tuple[int, int, int],
    goal_pose: tuple[int, int, int],
    config: PlannerConfig,
    course_cache: dict[tuple[CoursePortal, CoursePortal], PlanResult | None] | None = None,
) -> HybridPlan:
    config = resolve_course_config(map_data, config)
    grid_maze = extract_grid_maze(map_data, config)
    for name, pose in (("Start", start_pose), ("Goal", goal_pose)):
        row, column, direction = pose
        if not (0 <= row < MAZE_CELLS and 0 <= column < MAZE_CELLS and 0 <= direction < 4):
            raise ValueError(f"{name} pose is outside the 9 x 9 maze")
        if grid_maze.blocked_cells[row, column]:
            raise ValueError(f"{name} must be a standard maze cell outside the 5 x 5 course")

    detected = detect_course_portals(map_data, config)
    portals = [_portal_pose(portal, config) for portal in detected]
    candidates: list[tuple[tuple[float, int, float], HybridPlan]] = []
    cache = {} if course_cache is None else course_cache

    direct, direct_cells = plan_grid_commands(grid_maze, start_pose, goal_pose)
    direct_distance = math.inf
    direct_score = math.inf
    if direct_cells:
        direct_plan = HybridPlan(
            map_data,
            grid_maze,
            config,
            start_pose,
            goal_pose,
            None,
            None,
            direct,
            "",
            direct_cells,
            [],
            None,
            np.empty((0, 0, 3), dtype=np.uint8),
            np.empty((0, 0, 3), dtype=np.uint8),
        )
        direct_distance = direct.count("f") * CELL_MM
        direct_turns = direct.count("l") + direct.count("r")
        direct_score = direct_distance + config.route_turn_penalty_mm * direct_turns
        candidates.append(((direct_score, direct_turns, direct_distance), direct_plan))

    continuous_map = _course_map(map_data, config)
    for entry in portals:
        for exit_portal in portals:
            if entry == exit_portal:
                continue
            before_goal = (*entry.outside_cell, entry.inward_direction)
            after_start = (*exit_portal.outside_cell, exit_portal.outward_direction)
            before, cells_before = plan_grid_commands(grid_maze, start_pose, before_goal)
            after, cells_after = plan_grid_commands(grid_maze, after_start, goal_pose)
            if not cells_before or not cells_after:
                continue
            grid_distance = (before.count("f") + after.count("f") + 2) * CELL_MM
            grid_turns = (
                before.count("l") + before.count("r")
                + after.count("l") + after.count("r")
            )
            course_minimum = math.dist(
                _cell_centre(entry.inside_cell),
                _cell_centre(exit_portal.inside_cell),
            )
            optimistic = (
                grid_distance + course_minimum
                + config.route_turn_penalty_mm * grid_turns
            )
            if direct_score <= optimistic:
                continue
            selected = replace(
                config,
                start_heading_deg=DIRECTION_HEADINGS[entry.inward_direction],
                goal_heading_deg=DIRECTION_HEADINGS[exit_portal.outward_direction],
            )
            key = (entry.portal, exit_portal.portal)
            if key not in cache:
                try:
                    cache[key] = _plan_on_map(
                        continuous_map,
                        _cell_centre(entry.inside_cell),
                        _cell_centre(exit_portal.inside_cell),
                        selected,
                        render=False,
                    )
                except (RuntimeError, ValueError, AssertionError):
                    cache[key] = None
            continuous = cache[key]
            if continuous is None:
                continue
            distance = grid_distance + validate_result(continuous)["path_length_mm"]
            turns = grid_turns + sum(
                motion.turn_deg != 0 for motion in continuous.motions
            )
            score = distance + config.route_turn_penalty_mm * turns
            plan = HybridPlan(
                map_data,
                grid_maze,
                config,
                start_pose,
                goal_pose,
                entry,
                exit_portal,
                before,
                after,
                cells_before,
                cells_after,
                continuous,
                np.empty((0, 0, 3), dtype=np.uint8),
                np.empty((0, 0, 3), dtype=np.uint8),
            )
            candidates.append(((score, turns, distance), plan))
    if not candidates:
        raise RuntimeError(
            "No route exists through the detected outside walls or the 5 x 5 course"
        )
    result = min(candidates, key=lambda item: item[0])[1]
    result.occupancy_image = draw_hybrid_occupancy(result)
    result.route_image = draw_hybrid_route(result)
    validate_hybrid_result(result)
    return result


def plan_hybrid_route(
    image_path: str | Path,
    start_pose: tuple[int, int, int],
    goal_pose: tuple[int, int, int],
    config: PlannerConfig = PlannerConfig(),
) -> HybridPlan:
    rectified = rectify_maze(read_image(image_path))
    return plan_hybrid_on_map(
        extract_global_map(rectified, config), start_pose, goal_pose, config
    )


def plan_route(
    image_path: str | Path,
    start_mm: tuple[float, float],
    goal_mm: tuple[float, float],
    config: PlannerConfig = PlannerConfig(),
) -> PlanResult:
    rectified = rectify_maze(read_image(image_path))
    return _plan_on_map(extract_global_map(rectified, config), start_mm, goal_mm, config)

## 11. 绘图与指令生成

- `draw_hybrid_occupancy` / `draw_hybrid_route`：画占据地图和路线图。
- `hybrid_route_tokens`：生成 `BEGIN; ... END;` 指令串。三种记号：
  `T` 整数度转弯、`G180` 一个标准格子（Arduino 端带贴墙修正）、
  `F` 只在 5x5 场地内使用的连续直行距离。
- `validate_hybrid_result`：最后一道校验，检查每一段是否仍然避开占据区、
  终点误差是否在 10 mm 以内。

In [ ]:
from __future__ import annotations

# ---- 11.1 绘图 ----
def _draw_pose(
    image: np.ndarray,
    point: tuple[float, float],
    heading: int,
    colour: tuple[int, int, int],
    label: str,
    scale: float = 1.0,
) -> None:
    start = (int(round(point[0] * scale)), int(round(point[1] * scale)))
    tip = _endpoint(point, heading, 65.0 / scale)
    end = (int(round(tip[0] * scale)), int(round(tip[1] * scale)))
    thickness = max(3, int(round(5 * scale)))
    cv2.circle(image, start, max(6, int(round(10 * scale))), colour, -1, cv2.LINE_AA)
    cv2.arrowedLine(image, start, end, colour, thickness, cv2.LINE_AA, tipLength=0.30)
    cv2.putText(
        image,
        label,
        (start[0] + 12, start[1] - 12),
        cv2.FONT_HERSHEY_SIMPLEX,
        max(0.42, 0.65 * scale),
        colour,
        max(1, int(round(2 * scale))),
        cv2.LINE_AA,
    )


def _draw_grid(image: np.ndarray) -> None:
    for index in range(MAZE_CELLS + 1):
        coordinate = min(MAZE_MM - 1, index * CELL_MM)
        cv2.line(image, (coordinate, 0), (coordinate, MAZE_MM - 1), (125, 130, 135), 2)
        cv2.line(image, (0, coordinate), (MAZE_MM - 1, coordinate), (125, 130, 135), 2)


def draw_occupancy(map_data: GlobalMap, config: PlannerConfig) -> np.ndarray:
    view = map_data.rectified.copy()
    overlay = view.copy()
    overlay[map_data.occupancy_mask > 0] = (0, 150, 255)
    view = cv2.addWeighted(view, 0.58, overlay, 0.42, 0)
    wall_overlay = view.copy()
    wall_overlay[map_data.wall_mask > 0] = (0, 30, 230)
    view = cv2.addWeighted(view, 0.70, wall_overlay, 0.30, 0)
    for index, (x, y) in enumerate(map_data.obstacles_mm, 1):
        centre = (round(x), round(y))
        cv2.circle(view, centre, round(config.obstacle_diameter_mm / 2), (0, 0, 255), 5)
        cv2.putText(
            view, f"O{index}", (centre[0] + 12, centre[1] - 12),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2, cv2.LINE_AA,
        )
    _draw_grid(view)
    return view


def draw_route(
    map_data: GlobalMap,
    points: Sequence[tuple[float, float]],
    motions: Sequence[Motion],
    config: PlannerConfig,
) -> np.ndarray:
    view = draw_occupancy(map_data, config)
    pixel_points = [(round(x), round(y)) for x, y in points]
    driven = [motion for motion in motions if motion.distance_mm > 0]
    for index, (start, end, motion) in enumerate(
        zip(pixel_points, pixel_points[1:], driven), 1
    ):
        cv2.arrowedLine(view, start, end, (255, 70, 30), 7, cv2.LINE_AA, tipLength=0.04)
        midpoint = ((start[0] + end[0]) // 2, (start[1] + end[1]) // 2)
        cv2.putText(
            view,
            f"{index}: {motion.heading_deg}deg/{motion.distance_mm}mm",
            midpoint,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.50,
            (255, 70, 30),
            2,
            cv2.LINE_AA,
        )
    for index, point in enumerate(pixel_points[1:-1], 1):
        cv2.circle(view, point, 8, (255, 255, 0), -1, cv2.LINE_AA)
        cv2.putText(
            view, str(index), (point[0] + 10, point[1] - 8),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2, cv2.LINE_AA,
        )
    _draw_pose(view, points[0], config.start_heading_deg, (40, 220, 40), "S")
    final_heading = (
        config.goal_heading_deg
        if config.goal_heading_deg is not None
        else motions[-1].heading_deg if motions else config.start_heading_deg
    )
    _draw_pose(view, points[-1], final_heading, (220, 40, 220), "G")
    return view


def _course_cell(row: int, column: int, config: PlannerConfig) -> bool:
    return (
        config.course_top_row <= row < config.course_top_row + 5
        and config.course_left_column <= column < config.course_left_column + 5
    )


def _draw_detected_grid_walls(
    image: np.ndarray,
    maze: GridMaze,
    config: PlannerConfig,
) -> None:
    colour = (20, 35, 235)
    for row in range(MAZE_CELLS + 1):
        for column in range(MAZE_CELLS):
            adjacent = ((row - 1, column), (row, column))
            if any(_course_cell(r, c, config) for r, c in adjacent):
                continue
            if maze.horizontal_walls[row, column]:
                y = min(row * CELL_MM, MAZE_MM - 1)
                cv2.line(
                    image,
                    (column * CELL_MM, y),
                    ((column + 1) * CELL_MM, y),
                    colour,
                    8,
                    cv2.LINE_AA,
                )
    for row in range(MAZE_CELLS):
        for column in range(MAZE_CELLS + 1):
            adjacent = ((row, column - 1), (row, column))
            if any(_course_cell(r, c, config) for r, c in adjacent):
                continue
            if maze.vertical_walls[row, column]:
                x = min(column * CELL_MM, MAZE_MM - 1)
                cv2.line(
                    image,
                    (x, row * CELL_MM),
                    (x, (row + 1) * CELL_MM),
                    colour,
                    8,
                    cv2.LINE_AA,
                )


def _draw_course_boundary(
    image: np.ndarray,
    portals: Sequence[CoursePortal],
    config: PlannerConfig,
) -> None:
    top = config.course_top_row * CELL_MM
    left = config.course_left_column * CELL_MM
    bottom, right = top + 5 * CELL_MM, left + 5 * CELL_MM
    openings = {(portal.side, portal.offset) for portal in portals}
    for side in DIRECTIONS:
        for offset in range(5):
            if side == "N":
                start = (left + offset * CELL_MM, top)
                end = (left + (offset + 1) * CELL_MM, top)
            elif side == "E":
                start = (right, top + offset * CELL_MM)
                end = (right, top + (offset + 1) * CELL_MM)
            elif side == "S":
                start = (left + offset * CELL_MM, bottom)
                end = (left + (offset + 1) * CELL_MM, bottom)
            else:
                start = (left, top + offset * CELL_MM)
                end = (left, top + (offset + 1) * CELL_MM)
            if (side, offset) not in openings:
                cv2.line(image, start, end, (0, 215, 255), 8, cv2.LINE_AA)
                continue
            centre = ((start[0] + end[0]) // 2, (start[1] + end[1]) // 2)
            cv2.circle(image, centre, 13, (40, 220, 40), 4, cv2.LINE_AA)
            cv2.putText(
                image, f"{side}{offset}", (centre[0] + 15, centre[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, (40, 220, 40), 2, cv2.LINE_AA,
            )


def _hybrid_map_view(
    map_data: GlobalMap,
    maze: GridMaze,
    config: PlannerConfig,
) -> np.ndarray:
    view = map_data.rectified.copy()
    top = config.course_top_row * CELL_MM
    left = config.course_left_column * CELL_MM
    bottom, right = top + 5 * CELL_MM, left + 5 * CELL_MM
    course = view[top:bottom, left:right]
    overlay = course.copy()
    course_occupancy = _course_map(map_data, config).occupancy_mask
    occupied = course_occupancy[top:bottom, left:right] > 0
    overlay[occupied] = (0, 150, 255)
    view[top:bottom, left:right] = cv2.addWeighted(course, 0.60, overlay, 0.40, 0)
    _draw_grid(view)
    _draw_detected_grid_walls(view, maze, config)
    _draw_course_boundary(view, detect_course_portals(map_data, config), config)
    for index, (x, y) in enumerate(map_data.obstacles_mm, 1):
        if not (left <= x <= right and top <= y <= bottom):
            continue
        centre = (round(x), round(y))
        cv2.circle(view, centre, round(config.obstacle_diameter_mm / 2), (0, 0, 255), 5)
        cv2.putText(
            view, f"O{index}", (centre[0] + 12, centre[1] - 12),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2, cv2.LINE_AA,
        )
    return view


def draw_hybrid_occupancy(result: HybridPlan) -> np.ndarray:
    view = _hybrid_map_view(result.map_data, result.grid_maze, result.config)
    if result.uses_course:
        assert result.entry is not None and result.exit is not None
        for label, portal, colour in (
            ("P1", result.entry, (40, 220, 40)),
            ("P2", result.exit, (220, 40, 220)),
        ):
            point = tuple(round(value) for value in _cell_centre(portal.inside_cell))
            cv2.circle(view, point, 18, colour, 5, cv2.LINE_AA)
            cv2.putText(
                view, label, (point[0] + 22, point[1] - 16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, colour, 2, cv2.LINE_AA,
            )
    return view


def _draw_cell_path(
    image: np.ndarray,
    cells: Sequence[tuple[int, int]],
    colour: tuple[int, int, int],
) -> None:
    points = [tuple(round(value) for value in _cell_centre(cell)) for cell in cells]
    for start, end in zip(points, points[1:]):
        cv2.arrowedLine(image, start, end, colour, 9, cv2.LINE_AA, tipLength=0.10)


def draw_hybrid_route(result: HybridPlan) -> np.ndarray:
    view = draw_hybrid_occupancy(result)
    _draw_cell_path(view, result.cells_before, (40, 210, 40))
    _draw_cell_path(view, result.cells_after, (220, 170, 30))

    if not result.uses_course:
        _draw_pose(
            view, _cell_centre(result.start_pose[:2]),
            DIRECTION_HEADINGS[result.start_pose[2]], (40, 220, 40), "S",
        )
        _draw_pose(
            view, _cell_centre(result.goal_pose[:2]),
            DIRECTION_HEADINGS[result.goal_pose[2]], (220, 40, 220), "G",
        )
        cv2.putText(
            view, "GRID WALL ROUTE", (25, 55), cv2.FONT_HERSHEY_SIMPLEX,
            0.78, (40, 210, 40), 2, cv2.LINE_AA,
        )
        return view

    assert result.entry is not None and result.exit is not None
    assert result.continuous is not None

    entry_outside = tuple(round(value) for value in _cell_centre(result.entry.outside_cell))
    entry_inside = tuple(round(value) for value in _cell_centre(result.entry.inside_cell))
    exit_inside = tuple(round(value) for value in _cell_centre(result.exit.inside_cell))
    exit_outside = tuple(round(value) for value in _cell_centre(result.exit.outside_cell))
    cv2.arrowedLine(view, entry_outside, entry_inside, (40, 210, 40), 9, cv2.LINE_AA, tipLength=0.10)

    points = [tuple(round(value) for value in point) for point in result.continuous.waypoints_mm]
    for start, end in zip(points, points[1:]):
        cv2.arrowedLine(view, start, end, (255, 70, 30), 9, cv2.LINE_AA, tipLength=0.05)
    for index, point in enumerate(points[1:-1], 1):
        cv2.circle(view, point, 8, (255, 255, 0), -1, cv2.LINE_AA)
        cv2.putText(
            view, str(index), (point[0] + 10, point[1] - 8),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2, cv2.LINE_AA,
        )
    cv2.arrowedLine(view, exit_inside, exit_outside, (220, 170, 30), 9, cv2.LINE_AA, tipLength=0.10)

    _draw_pose(
        view,
        _cell_centre(result.start_pose[:2]),
        DIRECTION_HEADINGS[result.start_pose[2]],
        (40, 220, 40),
        "S",
    )
    _draw_pose(
        view,
        _cell_centre(result.goal_pose[:2]),
        DIRECTION_HEADINGS[result.goal_pose[2]],
        (220, 40, 220),
        "G",
    )
    cv2.putText(view, "GRID", entry_outside, cv2.FONT_HERSHEY_SIMPLEX, 0.65, (40, 210, 40), 2)
    course_label = (result.config.course_left_column * CELL_MM + 25,
                    result.config.course_top_row * CELL_MM + 45)
    cv2.putText(view, "CONTINUOUS 5x5", course_label, cv2.FONT_HERSHEY_SIMPLEX, 0.75,
                (255, 70, 30), 2, cv2.LINE_AA)
    cv2.putText(view, "GRID", exit_outside, cv2.FONT_HERSHEY_SIMPLEX, 0.65, (220, 170, 30), 2)
    return view

In [ ]:
from __future__ import annotations

# ---- 11.2 指令串、校验、文字说明、保存 ----
def navigation_steps(result: PlanResult) -> list[str]:
    x, y = result.control_points_mm[0]
    lines = [
        f"START  x={x:.1f} mm, y={y:.1f} mm, heading={result.start_heading_deg:+d} deg"
    ]
    action = 1
    segment = 0
    for motion in result.motions:
        if motion.turn_deg:
            lines.append(
                f"STEP {action:02d}  TURN  {motion.turn_deg:+d} deg"
                f" -> heading {motion.heading_deg:+d} deg"
            )
            action += 1
        if motion.distance_mm:
            segment += 1
            x, y = result.waypoints_mm[segment]
            lines.append(
                f"STEP {action:02d}  DRIVE {motion.distance_mm:d} mm"
                f" -> x={x:.1f} mm, y={y:.1f} mm"
            )
            action += 1
    final_heading = (
        result.goal_heading_deg
        if result.goal_heading_deg is not None
        else result.motions[-1].heading_deg if result.motions else result.start_heading_deg
    )
    lines.append(f"GOAL   x={x:.1f} mm, y={y:.1f} mm, heading={final_heading:+d} deg")
    return lines


def validate_result(result: PlanResult) -> dict[str, float]:
    if len(result.waypoints_mm) < 2:
        raise AssertionError("Route has no straight segment")
    driven = [motion for motion in result.motions if motion.distance_mm > 0]
    if len(driven) != len(result.waypoints_mm) - 1:
        raise AssertionError("Motion count does not match waypoint count")
    path_length = 0.0
    for start, end, motion in zip(result.waypoints_mm, result.waypoints_mm[1:], driven):
        if not isinstance(motion.turn_deg, int) or not isinstance(motion.heading_deg, int):
            raise AssertionError("A turn or heading is not an integer")
        if not 1 <= motion.distance_mm <= result.config.maximum_segment_mm + 2:
            raise AssertionError("A straight segment is outside Arduino limits")
        if not _segment_clear(start, end, result.map_data.occupancy_mask):
            raise AssertionError("Rounded route crosses an occupied safety area")
        path_length += math.hypot(end[0] - start[0], end[1] - start[1])
    goal = result.control_points_mm[-1]
    predicted = result.waypoints_mm[-1]
    goal_error = math.dist(goal, predicted)
    if goal_error > 10.0:
        raise AssertionError("Integer-heading route finishes more than 10 mm from the goal")
    return {
        "path_length_mm": path_length,
        "goal_error_mm": goal_error,
    }


def _grid_tokens(commands: str) -> list[str]:
    tokens: list[str] = []
    for command in commands:
        if command == "f":
            tokens.append(f"G{CELL_MM};")
        elif command == "l":
            tokens.append("T90;")
        elif command == "r":
            tokens.append("T-90;")
        else:
            raise ValueError(f"Unknown grid command: {command!r}")
    return tokens


def hybrid_route_tokens(result: HybridPlan) -> list[str]:
    if not result.uses_course:
        return ["BEGIN;", *_grid_tokens(result.grid_before), "END;"]
    assert result.continuous is not None
    tokens = ["BEGIN;", *_grid_tokens(result.grid_before), f"G{CELL_MM};"]
    for motion in result.continuous.motions:
        if motion.turn_deg:
            tokens.append(f"T{motion.turn_deg};")
        if motion.distance_mm:
            tokens.append(f"F{motion.distance_mm};")
    tokens.extend((f"G{CELL_MM};", *_grid_tokens(result.grid_after), "END;"))
    return tokens


def hybrid_arduino_declaration(result: HybridPlan) -> str:
    return f'#define GENERATED_ROUTE "{"".join(hybrid_route_tokens(result))}"'


def _grid_pose_text(pose: tuple[int, int, int]) -> str:
    return f"({pose[0]},{pose[1]},{DIRECTIONS[pose[2]]})"


def validate_hybrid_result(result: HybridPlan) -> dict[str, float]:
    if not result.cells_before or result.cells_before[0] != result.start_pose[:2]:
        raise AssertionError("GRID path does not start at the selected pose")
    if not result.uses_course:
        if result.cells_before[-1] != result.goal_pose[:2]:
            raise AssertionError("GRID path does not reach the selected goal")
        forwards = result.grid_before.count("f")
        return {
            "path_length_mm": float(forwards * CELL_MM),
            "goal_error_mm": 0.0,
            "grid_cells": float(forwards),
        }

    assert result.entry is not None and result.exit is not None
    assert result.continuous is not None
    metrics = validate_result(result.continuous)
    if result.cells_before[-1] != result.entry.outside_cell:
        raise AssertionError("GRID-before path does not reach the course entrance")
    if result.cells_after[0] != result.exit.outside_cell:
        raise AssertionError("GRID-after path does not start at the course exit")
    if result.cells_after[-1] != result.goal_pose[:2]:
        raise AssertionError("GRID-after path does not reach the selected goal")
    top = result.config.course_top_row * CELL_MM
    left = result.config.course_left_column * CELL_MM
    for x, y in result.continuous.waypoints_mm:
        if not (
            left <= x <= left + COURSE_CELLS * CELL_MM
            and top <= y <= top + COURSE_CELLS * CELL_MM
        ):
            raise AssertionError("A continuous waypoint left the 5 x 5 course")
    metrics["grid_cells"] = float(
        result.grid_before.count("f") + result.grid_after.count("f") + 2
    )
    return metrics


def hybrid_navigation_steps(result: HybridPlan) -> list[str]:
    if not result.uses_course:
        return [
            f"START {_grid_pose_text(result.start_pose)}",
            f"GRID WALL ROUTE: {result.grid_before or '(already at goal pose)'}",
            f"GOAL {_grid_pose_text(result.goal_pose)}",
        ]
    assert result.entry is not None and result.exit is not None
    assert result.continuous is not None
    lines = [
        f"START {_grid_pose_text(result.start_pose)}",
        f"GRID BEFORE: {result.grid_before or '(already at entrance)'}",
        f"ENTER: G{CELL_MM} to cell {result.entry.inside_cell}",
        "CONTINUOUS 5x5:",
        *[f"  {line}" for line in navigation_steps(result.continuous)],
        f"EXIT: G{CELL_MM} to cell {result.exit.outside_cell}",
        f"GRID AFTER: {result.grid_after or '(already at goal)'}",
        f"GOAL {_grid_pose_text(result.goal_pose)}",
    ]
    return lines


def save_hybrid_result(result: HybridPlan, output_dir: str | Path) -> Path:
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    write_image(output / "rectified_maze.png", result.map_data.rectified)
    write_image(output / "hybrid_occupancy_map.png", result.occupancy_image)
    write_image(output / "hybrid_planned_route.png", result.route_image)
    route_file = output / "route_commands.txt"
    entry_text = exit_text = "not used"
    if result.uses_course:
        assert result.entry is not None and result.exit is not None
        entry_text = (
            f"{result.entry.portal.side}{result.entry.portal.offset}, "
            f"inside {result.entry.inside_cell}"
        )
        exit_text = (
            f"{result.exit.portal.side}{result.exit.portal.offset}, "
            f"inside {result.exit.inside_cell}"
        )
    lines = [
        f"START = {_grid_pose_text(result.start_pose)}",
        f"GOAL = {_grid_pose_text(result.goal_pose)}",
        f"MODE = {'GRID + CONTINUOUS 5x5' if result.uses_course else 'GRID WALL ROUTE'}",
        f"COURSE = top-left ({result.config.course_top_row}, {result.config.course_left_column}), size 5 x 5",
        "DETECTED PORTALS = " + ", ".join(
            f"{portal.side}{portal.offset}"
            for portal in detect_course_portals(result.map_data, result.config)
        ),
        f"ENTRY = {entry_text}",
        f"EXIT = {exit_text}",
        "",
        "# Complete navigation",
        *hybrid_navigation_steps(result),
        "",
        "# Paste between the GENERATED ROUTE markers in the Arduino sketch",
        hybrid_arduino_declaration(result),
        "",
        "# G = normal 180 mm grid cell with wall following",
        "# F = continuous-course straight distance without wall following",
        " ".join(hybrid_route_tokens(result)),
    ]
    route_file.write_text("\n".join(lines) + "\n", encoding="ascii")
    return route_file


def print_hybrid_result(result: HybridPlan) -> None:
    metrics = validate_hybrid_result(result)
    print(f"Start: {_grid_pose_text(result.start_pose)}")
    print(f"Goal:  {_grid_pose_text(result.goal_pose)}")
    print(f"Mode:  {'GRID + CONTINUOUS 5x5' if result.uses_course else 'GRID WALL ROUTE'}")
    print("Detected portals: " + ", ".join(
        f"{portal.side}{portal.offset}"
        for portal in detect_course_portals(result.map_data, result.config)
    ))
    if result.uses_course:
        assert result.entry is not None and result.exit is not None
        print(f"Course entry: {result.entry.portal.side}{result.entry.portal.offset}")
        print(f"Course exit:  {result.exit.portal.side}{result.exit.portal.offset}")
    print(f"GRID cells: {int(metrics['grid_cells'])}")
    if result.uses_course:
        print(f"Continuous path: {metrics['path_length_mm']:.1f} mm")
        print(f"Continuous goal error: {metrics['goal_error_mm']:.1f} mm")
    print("\n".join(hybrid_navigation_steps(result)))
    print("\nPaste into Arduino:")
    print(hybrid_arduino_declaration(result))

## 12. 拍照 / 读图

三种来源统一在 `load_source_image()` 里处理。

**摄像头模式**会弹出一个实时预览窗口：

| 按键 | 作用 |
| --- | --- |
| `空格` 或 `回车` | 拍照，存盘并立刻进入识别 |
| `N` | 切换到下一个摄像头（找外接摄像头就按这个） |
| `ESC` 或 `Q` | 放弃拍照 |

预览窗口是本机桌面窗口，所以摄像头模式只能在**本地运行**的 Jupyter 里用
（远程服务器 / Colab 上没有摄像头窗口）。

In [ ]:
from __future__ import annotations

import datetime
import sys

def list_available_cameras(max_index: int = 4) -> list[int]:
    """探测哪些摄像头序号可用。Windows 上用 DirectShow 后端，打开快很多。"""
    backend = cv2.CAP_DSHOW if sys.platform.startswith("win") else cv2.CAP_ANY
    found = []
    for index in range(max_index):
        capture = cv2.VideoCapture(index, backend)
        if capture.isOpened():
            ok, _ = capture.read()
            if ok:
                found.append(index)
        capture.release()
    return found


def capture_from_camera(
    index: int = 0,
    save_dir: str | Path = "captures",
    width: int = 1920,
    height: int = 1080,
) -> Path | None:
    """打开预览窗口拍一张照片并存盘，返回保存路径；放弃则返回 None。"""
    backend = cv2.CAP_DSHOW if sys.platform.startswith("win") else cv2.CAP_ANY
    window = "Maze capture - SPACE=shoot  N=next camera  ESC=cancel"

    def open_camera(camera_index: int):
        capture = cv2.VideoCapture(camera_index, backend)
        if not capture.isOpened():
            capture.release()
            return None
        # 尽量用高分辨率拍，识别靠的是细节。相机不支持就自动退回默认值。
        capture.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        capture.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
        return capture

    current = index
    capture = open_camera(current)
    if capture is None:
        available = list_available_cameras()
        raise RuntimeError(
            f"打不开摄像头 {current}。可用的序号：{available if available else '一个都没找到'}"
        )

    saved: Path | None = None
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window, 960, 540)
    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                print(f"摄像头 {current} 读不到画面，尝试下一个。")
                capture.release()
                current = (current + 1) % 5
                capture = open_camera(current)
                if capture is None:
                    raise RuntimeError("没有可用的摄像头")
                continue

            preview = frame.copy()
            resolution = f"{frame.shape[1]}x{frame.shape[0]}"
            for line, text in enumerate((
                f"camera {current}   {resolution}",
                "SPACE / ENTER = shoot",
                "N = next camera    ESC / Q = cancel",
            )):
                # 先描黑边再写白字，保证在任何背景上都看得清。
                position = (12, 30 + line * 26)
                cv2.putText(preview, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, (0, 0, 0), 4, cv2.LINE_AA)
                cv2.putText(preview, text, position, cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.imshow(window, preview)

            key = cv2.waitKey(30) & 0xFF
            if key in (27, ord("q"), ord("Q")):        # ESC / Q 放弃
                print("已取消拍照。")
                break
            if key in (ord("n"), ord("N")):            # N 换摄像头
                capture.release()
                current = (current + 1) % 5
                nxt = open_camera(current)
                while nxt is None:
                    current = (current + 1) % 5
                    if current == index:
                        raise RuntimeError("没有其它可用的摄像头")
                    nxt = open_camera(current)
                capture = nxt
                print(f"切换到摄像头 {current}")
                continue
            if key in (32, 13):                        # 空格 / 回车 拍照
                folder = Path(save_dir)
                folder.mkdir(parents=True, exist_ok=True)
                stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                saved = folder / f"maze_{stamp}.jpg"
                # 存原始帧，不是带提示文字的预览帧。
                cv2.imwrite(str(saved), frame)
                print(f"已保存：{saved.resolve()}  ({resolution})")
                break
    finally:
        capture.release()
        cv2.destroyWindow(window)
        # OpenCV 的窗口要多跑几次事件循环才会真正消失。
        for _ in range(5):
            cv2.waitKey(1)
    return saved


def load_source_image() -> tuple[np.ndarray, str]:
    """按 IMAGE_SOURCE 取一张 BGR 图，返回 (图像, 来源说明)。"""
    if IMAGE_SOURCE == "camera":
        path = capture_from_camera(CAMERA_INDEX, CAPTURE_DIR)
        if path is None:
            raise RuntimeError("没有拍照，无法继续。把 IMAGE_SOURCE 改成 'demo' 也可以先跑通流程。")
        return read_image(path), str(path)

    if IMAGE_SOURCE == "file":
        if not IMAGE_PATH:
            raise ValueError("IMAGE_SOURCE = 'file' 时必须填 IMAGE_PATH")
        return read_image(IMAGE_PATH), str(IMAGE_PATH)

    if IMAGE_SOURCE == "demo":
        raw = np.frombuffer(base64.b64decode(DEMO_IMAGE_BASE64), dtype=np.uint8)
        image = cv2.imdecode(raw, cv2.IMREAD_COLOR)
        if image is None:
            raise RuntimeError("内嵌示例照片解码失败")
        return image, "内嵌示例照片 (c10.jpg)"

    raise ValueError(f"IMAGE_SOURCE 只能是 'demo' / 'camera' / 'file'，收到 {IMAGE_SOURCE!r}")


print("可用摄像头探测放在下一格，避免每次 Run All 都去开一次相机。")

### 12.1 （可选）先看看有哪些摄像头

要用外接摄像头但不知道序号是几，就跑这一格。探测会逐个尝试打开设备，
大概要几秒钟。

In [ ]:
# 只有需要找摄像头序号时才跑这一格。
print("正在探测摄像头，请稍候……")
cameras = list_available_cameras(max_index=4)
if cameras:
    print("可用摄像头序号：", cameras)
    print("把上面第 2 节的 CAMERA_INDEX 改成你要的那个；或者在预览窗口里按 N 现场切换。")
else:
    print("没有探测到摄像头。检查一下是否被其它程序占用，或者系统的相机权限。")

## 13. 运行识别与规划

这一格就是主流程：取图 -> 矫正 -> 识别墙和圆柱 -> 找门洞 -> 规划 -> 输出。

`IMAGE_SOURCE = "camera"` 时，这一格会先弹出预览窗口让你拍照，拍完存盘并**自动**
接着做识别，不需要再手动指定文件名。

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt

# 朝向字母 -> 内部编号（0=N, 1=E, 2=S, 3=W）
def to_pose(pose: tuple[int, int, str | int]) -> tuple[int, int, int]:
    row, column, heading = pose
    if isinstance(heading, str):
        heading = DIRECTIONS.index(heading.upper())
    return (int(row), int(column), int(heading))


config = PlannerConfig(
    course_top_row=COURSE_TOP_ROW,
    course_left_column=COURSE_LEFT_COLUMN,
)

# ---- 第 1 步：取图 -----------------------------------------------------
source_image, source_label = load_source_image()
print(f"照片来源：{source_label}   尺寸 {source_image.shape[1]} x {source_image.shape[0]}")

# ---- 第 2 步：透视矫正 -------------------------------------------------
rectified = rectify_maze(source_image)
print(f"矫正完成：{rectified.shape[1]} x {rectified.shape[0]} px（1 px = 1 mm）")

# 先把矫正结果显示出来。照片拍歪了或者角标没识别到，在这里一眼就能看出来，
# 不用等到后面报一个看不懂的错。
preview, panes = plt.subplots(1, 2, figsize=(12, 5.4))
panes[0].imshow(source_image[:, :, ::-1])
panes[0].set_title("Source photo")
panes[1].imshow(rectified[:, :, ::-1])
panes[1].set_title("Rectified (should be a square 9x9 grid)")
for pane in panes:
    pane.set_xticks([])
    pane.set_yticks([])
preview.tight_layout()
plt.show()

# ---- 第 3 步：识别墙体和圆柱，并自动定位 5x5 场地 ----------------------
try:
    map_data = extract_global_map(rectified, config)
except ValueError as error:
    print("识别失败：", error)
    print("常见原因：迷宫被人或杂物挡住太多。请从正上方重拍一张再试。")
    raise
except RuntimeError as error:
    print("场地定位失败：")
    print(error)
    raise

# 把自动找到的场地位置写回 config，后面所有步骤都用这份具体的配置。
config = resolve_course_config(map_data, config)
print(f"5x5 场地：左上角 (行 {map_data.course_top_row}, 列 {map_data.course_left_column})")
if map_data.course_candidates:
    # 场地内部是空的，普通迷宫格子里到处是墙 —— 这是最强的判据。
    print("  候选排名（interior 越小越像场地）：")
    for candidate in map_data.course_candidates:
        print(f"    {candidate.position}  score {candidate.score:.3f}"
              f"  interior {candidate.interior_wall:.3f}"
              f"  boundary {candidate.boundary_wall:.3f}"
              f"  圆柱 {candidate.cylinders_inside}/{candidate.cylinders_total}")
print(f"检测到圆柱 {len(map_data.obstacles_mm)} 个：",
      [(round(x), round(y)) for x, y in map_data.obstacles_mm])

# ---- 第 4 步：识别 5x5 场地的门洞 --------------------------------------
portals = detect_course_portals(map_data, config)
print("检测到门洞：", ", ".join(f"{p.side}{p.offset}" for p in portals) or "无")

# ---- 第 5 步：规划 -----------------------------------------------------
start_pose, goal_pose = to_pose(START), to_pose(GOAL)
try:
    result = plan_hybrid_on_map(map_data, start_pose, goal_pose, config)
except (ValueError, RuntimeError) as error:
    print("规划失败：", error)
    print("\n排查顺序：")
    print("  1. 上面那张矫正图是不是一个端正的 9x9 方格？不是的话重拍照片。")
    print("  2. START / GOAL 是不是落在 5x5 场地里面了？必须在场地外的普通格子。")
    print("  3. START / GOAL 是不是落在被切掉的墙角格子上了？")
    print("  4. COURSE_TOP_ROW / COURSE_LEFT_COLUMN 填的位置对不对？")
    print("  5. 门洞是不是一个都没识别到（上一行会显示“无”）？那多半是照片质量问题。")
    raise

print("\n" + "=" * 60)
print_hybrid_result(result)

## 14. 结果显示与校验

三张图：矫正后的迷宫、带安全膨胀的占据地图、规划出来的路线。
网格线每 180 mm 一条，正好对应一个格子。

In [ ]:
from __future__ import annotations

views = [
    ("Rectified 9x9 maze", rectified),
    ("Occupancy map (safety inflation inside 5x5)", draw_hybrid_occupancy(result)),
    ("Planned route", draw_hybrid_route(result)),
]

figure, axes = plt.subplots(1, len(views), figsize=(6.5 * len(views), 6.8))
for axis, (title, image) in zip(np.atleast_1d(axes), views):
    axis.imshow(image[:, :, ::-1])          # OpenCV 是 BGR，matplotlib 要 RGB
    axis.set_title(title, fontsize=11)
    axis.set_xlabel("x (mm)")
    axis.set_ylabel("y (mm)")
    axis.set_xticks(range(0, MAZE_MM + 1, CELL_MM))
    axis.set_yticks(range(0, MAZE_MM + 1, CELL_MM))
    axis.tick_params(labelsize=7)
    axis.grid(alpha=0.25)
figure.tight_layout()
plt.show()

# ---- 最后一道校验 ------------------------------------------------------
checks = validate_hybrid_result(result)
print("校验结果：")
for name, value in checks.items():
    print(f"  {name:24s} {value:.2f}")
print("\n（校验不通过会直接抛 AssertionError，能跑到这里就说明路线是安全的。）")

### 14.1 Arduino 指令串

把下面打印出来的那一整行，粘贴到
`week12_4_2_waypoint_runner.ino` 里 `PASTE GENERATED ROUTE` 两行注释之间。

记号含义：`T` = 整数度转弯（正数左转）、`G180` = 一个标准格子（带贴墙修正）、
`F` = 只在 5x5 场地内使用的连续直行距离。

In [ ]:
declaration = hybrid_arduino_declaration(result)
print(declaration)
print()
print("逐条动作：")
for step in hybrid_navigation_steps(result):
    print("  " + step)

# 想直接复制到剪贴板就把下面两行的注释去掉（需要图形界面）。
# import tkinter as tk
# root = tk.Tk(); root.withdraw(); root.clipboard_clear(); root.clipboard_append(declaration); root.update(); root.destroy()

### 14.2 （可选）把图和指令写成文件

只有设置了 `SAVE_OUTPUT_DIR` 才会写盘，默认不产生任何文件。

In [ ]:
if SAVE_OUTPUT_DIR:
    saved_path = save_hybrid_result(result, SAVE_OUTPUT_DIR)
    print("已保存到：", Path(SAVE_OUTPUT_DIR).resolve())
    print("指令文件：", saved_path)
else:
    print("SAVE_OUTPUT_DIR 是 None，本次不写任何文件（结果只在 notebook 里显示）。")

## 15.（可选）交互式选点窗口

前面是用第 2 节的 `START` / `GOAL` 变量直接规划，这样在任何环境下都能跑。
如果你更习惯用鼠标点，可以跑下面这一格，它会打开原来那个交互窗口：

- 在普通格子里**按住拖动**表示起点位姿（拖动方向 = 朝向），再拖一次表示终点；
- 也可以点一下格子然后按 `N` / `E` / `S` / `W`；
- `R` 重选、`C` 复制指令、`S` 保存、`ESC` 关闭。

同样只能在本地桌面环境使用。窗口打开期间内核是被占用的，关掉窗口才会继续。

In [ ]:
from __future__ import annotations

# ---- 15.1 交互窗口（可选）----
class ContinuousNavigatorUI:
    def __init__(
        self,
        image_path: str | Path,
        config: PlannerConfig = PlannerConfig(),
        output_dir: str | Path = "output",
    ) -> None:
        self.source = Path(image_path)
        self.config = config
        self.output_dir = Path(output_dir)
        self.original = read_image(self.source)
        self.map_data = extract_global_map(rectify_maze(self.original), config)
        self.grid_maze = extract_grid_maze(self.map_data, config)
        self.portals = detect_course_portals(self.map_data, config)
        self.course_cache: dict[
            tuple[CoursePortal, CoursePortal], PlanResult | None
        ] = {}
        self.occupancy = _hybrid_map_view(self.map_data, self.grid_maze, config)
        self.start: tuple[int, int, int] | None = None
        self.goal: tuple[int, int, int] | None = None
        self.pending_cell: tuple[int, int] | None = None
        self.drag: tuple[int, int, tuple[int, int]] | None = None
        self.result: HybridPlan | None = None
        self.solving = False
        self.status = "Set START in a standard maze cell"
        self.window = "Week 12 - Hybrid grid and continuous navigation"
        self.clipboard_root = None

    @staticmethod
    def _pose_text(pose: tuple[int, int, int] | None) -> str:
        if pose is None:
            return "-"
        return _grid_pose_text(pose)

    @staticmethod
    def _put(
        image: np.ndarray,
        text: str,
        x: int,
        y: int,
        scale: float = 0.50,
        colour: tuple[int, int, int] = (220, 225, 230),
        thickness: int = 1,
    ) -> None:
        cv2.putText(
            image, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
            scale, colour, thickness, cv2.LINE_AA,
        )

    @staticmethod
    def _screen_cell(x: int, y: int) -> tuple[int, int]:
        cell_px = DISPLAY_PX / MAZE_CELLS
        return (
            int(np.clip(y // cell_px, 0, MAZE_CELLS - 1)),
            int(np.clip(x // cell_px, 0, MAZE_CELLS - 1)),
        )

    def assign_pose(self, cell: tuple[int, int], direction: int) -> None:
        self.pending_cell = None
        if self.grid_maze.blocked_cells[cell]:
            self.status = "Choose a standard cell outside the 5 x 5 course"
            return
        pose = (cell[0], cell[1], direction % 4)
        if self.start is None:
            self.start = pose
            self.status = "Set GOAL in a standard maze cell"
        elif self.goal is None:
            self.goal = pose
            self.solve()
        else:
            self.start = pose
            self.goal = None
            self.result = None
            self.status = "New START set; now set GOAL"

    def solve(self) -> None:
        if self.solving:
            return
        assert self.start is not None and self.goal is not None
        self.solving = True
        try:
            try:
                result = plan_hybrid_on_map(
                    self.map_data,
                    self.start,
                    self.goal,
                    self.config,
                    self.course_cache,
                )
            except (RuntimeError, ValueError, AssertionError) as error:
                self.goal = None
                self.result = None
                self.status = f"No route: {error}"
                print(self.status)
                return
            route_file = save_hybrid_result(result, self.output_dir)
            self.result = result
            mode = "Hybrid" if result.uses_course else "Outside GRID"
            self.status = f"{mode} ROUTE READY - Esc closes the window"
            print_hybrid_result(result)
            print(f"Saved commands: {route_file.resolve()}", flush=True)
            self.copy_commands(result)
        finally:
            self.solving = False

    def reset(self) -> None:
        self.start = None
        self.goal = None
        self.pending_cell = None
        self.drag = None
        self.result = None
        self.status = "Set START in a standard maze cell"

    def copy_commands(self, result: HybridPlan | None = None) -> None:
        selected = self.result if result is None else result
        if selected is None:
            return
        try:
            if self.clipboard_root is None:
                import tkinter as tk

                self.clipboard_root = tk.Tk()
                self.clipboard_root.withdraw()
            text = hybrid_arduino_declaration(selected)
            self.clipboard_root.clipboard_clear()
            self.clipboard_root.clipboard_append(text)
            self.clipboard_root.update()
        except Exception:
            self.clipboard_root = None

    def save(self) -> None:
        self.output_dir.mkdir(parents=True, exist_ok=True)
        write_image(self.output_dir / "rectified_maze.png", self.map_data.rectified)
        write_image(self.output_dir / "hybrid_occupancy_map.png", self.occupancy)
        if self.result is not None:
            save_hybrid_result(self.result, self.output_dir)

    def mouse(self, event: int, x: int, y: int, _flags: int, _parameter) -> None:
        if self.solving:
            return
        if event == cv2.EVENT_RBUTTONUP:
            self.reset()
            return
        if event == cv2.EVENT_LBUTTONDOWN and 0 <= x < DISPLAY_PX and 0 <= y < DISPLAY_PX:
            self.drag = (x, y, self._screen_cell(x, y))
            return
        if event != cv2.EVENT_LBUTTONUP or self.drag is None:
            return
        start_x, start_y, cell = self.drag
        self.drag = None
        dx, dy = x - start_x, y - start_y
        if math.hypot(dx, dy) < 14:
            self.pending_cell = cell
            self.status = "Press N, E, S or W"
            return
        if abs(dx) >= abs(dy):
            direction = 1 if dx > 0 else 3
        else:
            direction = 2 if dy > 0 else 0
        self.assign_pose(cell, direction)

    @staticmethod
    def _wrapped_tokens(tokens: Sequence[str], width: int = 34) -> list[str]:
        lines: list[str] = []
        line = ""
        for token in tokens:
            candidate = token if not line else f"{line} {token}"
            if len(candidate) > width:
                lines.append(line)
                line = token
            else:
                line = candidate
        if line:
            lines.append(line)
        return lines

    def render(self) -> np.ndarray:
        source = self.result.route_image if self.result is not None else self.occupancy
        view = cv2.resize(source, (DISPLAY_PX, DISPLAY_PX), interpolation=cv2.INTER_AREA)
        scale = DISPLAY_PX / MAZE_MM
        if self.result is None and self.start is not None:
            _draw_pose(
                view,
                _cell_centre(self.start[:2]),
                DIRECTION_HEADINGS[self.start[2]],
                (40, 220, 40),
                "S",
                scale,
            )
        if self.pending_cell is not None:
            centre = tuple(round(value * scale) for value in _cell_centre(self.pending_cell))
            cv2.circle(view, centre, 11, (0, 210, 255), 3, cv2.LINE_AA)
        for row, column in BLOCKED_CORNERS:
            x0, y0 = round(column * CELL_MM * scale), round(row * CELL_MM * scale)
            x1, y1 = round((column + 1) * CELL_MM * scale), round((row + 1) * CELL_MM * scale)
            cv2.line(view, (x0 + 16, y0 + 16), (x1 - 16, y1 - 16), (185, 190, 195), 4)
            cv2.line(view, (x1 - 16, y0 + 16), (x0 + 16, y1 - 16), (185, 190, 195), 4)

        panel = np.full((DISPLAY_PX, PANEL_PX, 3), (29, 32, 36), dtype=np.uint8)
        self._put(panel, "WEEK 12 HYBRID NAVIGATION", 20, 38, 0.64, (245, 245, 245), 2)
        mode_text = "DETECTED WALL GRID + CONTINUOUS 5x5"
        if self.result is not None and not self.result.uses_course:
            mode_text = "DETECTED WALL GRID ROUTE"
        self._put(panel, mode_text, 20, 72, 0.43, (175, 185, 195))
        self._put(panel, f"Start: {self._pose_text(self.start)}", 20, 108, 0.48, (70, 220, 90), 2)
        self._put(panel, f"Goal:  {self._pose_text(self.goal)}", 20, 140, 0.48, (220, 90, 220), 2)
        self._put(
            panel,
            f"Cylinders: {len(self.map_data.obstacles_mm)}",
            20,
            174,
            0.46,
            (175, 185, 195),
        )
        portal_names = ",".join(
            f"{portal.side}{portal.offset}" for portal in self.portals
        ) or "none"
        portal_text = f"Open({len(self.portals)}): {portal_names}"
        self._put(panel, portal_text, 190, 174, 0.42, (175, 185, 195))
        self._put(panel, "Complete Arduino route", 20, 214, 0.54, (70, 205, 245), 2)

        tokens = ["-"]
        if self.result is not None:
            tokens = hybrid_route_tokens(self.result)
        lines = self._wrapped_tokens(tokens)
        y = 246
        for line in lines[:13]:
            self._put(panel, line, 20, y, 0.46, (245, 215, 85))
            y += 25
        if len(lines) > 13:
            self._put(panel, f"+ {len(lines) - 13} more lines in route_commands.txt", 20, y, 0.40)

        self._put(panel, "Drag in START cell, then GOAL cell", 20, 626, 0.44, (185, 192, 202))
        self._put(panel, "Click only: N/E/S/W    R: reset", 20, 653, 0.42, (185, 192, 202))
        self._put(panel, "C: copy code    S: save    Esc: close", 20, 678, 0.42, (185, 192, 202))
        self._put(panel, self.status[:50], 20, 707, 0.41, (90, 205, 245))
        return np.hstack((view, panel))

    def run(self) -> HybridPlan | None:
        cv2.namedWindow(self.window, cv2.WINDOW_AUTOSIZE)
        cv2.setMouseCallback(self.window, self.mouse)
        preview = self.original.copy()
        maximum = max(preview.shape[:2])
        if maximum > 900:
            preview = cv2.resize(
                preview, None, fx=900 / maximum, fy=900 / maximum,
                interpolation=cv2.INTER_AREA,
            )
        cv2.imshow("Week 12 - Original photo input", preview)
        self.save()
        cardinal = {
            ord("n"): 0, ord("N"): 0,
            ord("e"): 1, ord("E"): 1,
            ord("s"): 2, ord("S"): 2,
            ord("w"): 3, ord("W"): 3,
        }
        while True:
            cv2.imshow(self.window, self.render())
            key = cv2.waitKey(20) & 0xFF
            if cv2.getWindowProperty(self.window, cv2.WND_PROP_VISIBLE) < 1:
                break
            if key == 27:
                break
            if self.pending_cell is not None and key in cardinal:
                self.assign_pose(self.pending_cell, cardinal[key])
            elif key in (ord("r"), ord("R")):
                self.reset()
            elif key in (ord("c"), ord("C")):
                self.copy_commands()
            elif key in (ord("s"), ord("S")):
                self.save()
        cv2.destroyAllWindows()
        if self.clipboard_root is not None:
            self.clipboard_root.destroy()
            self.clipboard_root = None
        return self.result

In [ ]:
from __future__ import annotations

# 跑这一格会打开交互窗口。UI 需要一个图片路径，所以内嵌示例照片先写成临时文件。
import tempfile

def launch_interactive_ui() -> None:
    if IMAGE_SOURCE == "demo":
        folder = Path(tempfile.gettempdir())
        image_path = folder / "week12_demo_maze.jpg"
        image_path.write_bytes(base64.b64decode(DEMO_IMAGE_BASE64))
    elif IMAGE_SOURCE == "file":
        image_path = Path(IMAGE_PATH)
    else:
        captured = capture_from_camera(CAMERA_INDEX, CAPTURE_DIR)
        if captured is None:
            print("没有拍照，取消。")
            return
        image_path = captured

    output = Path(SAVE_OUTPUT_DIR) if SAVE_OUTPUT_DIR else Path(tempfile.gettempdir()) / "week12_ui_output"
    ContinuousNavigatorUI(image_path, config, output).run()

# 取消下面这行的注释来打开窗口。
# launch_interactive_ui()
print("交互窗口已就绪：取消上面 launch_interactive_ui() 的注释即可打开。")

## 16.（可选）通过串口直接下发给 Arduino

不想手动粘贴指令的话，可以让 notebook 直接把指令一条条发给板子。前提：

1. 烧录 `week12_4_2_waypoint_runner.ino` 时，`PASTE GENERATED ROUTE` 之间**留空**
   （这样板子会进入等待串口指令的模式）；
2. 关掉 Arduino IDE 的串口监视器，否则端口被占用；
3. 需要 `pyserial`。

板子每做完一条指令回 `DONE`，全部完成回 `FINISHED`，遇到障碍回 `BLOCKED` 并停止。

In [ ]:
from __future__ import annotations

# ---- 16.1 串口下发 ----
def send_to_arduino(
    port: str,
    result: HybridPlan,
    baudrate: int = 115200,
    ready_timeout_s: float = 20.0,
) -> None:
    try:
        import serial
    except ImportError as error:
        raise RuntimeError("Install pyserial in the Week 12 environment first") from error

    with serial.Serial(port, baudrate, timeout=0.2) as board:
        deadline = time.time() + ready_timeout_s
        while time.time() < deadline:
            if board.readline().decode(errors="ignore").strip() == "READY":
                break
        else:
            raise TimeoutError("Arduino did not report READY")

        for token in hybrid_route_tokens(result):
            board.write((token + "\n").encode("ascii"))
            board.flush()
            expected = "FINISHED" if token == "END;" else "DONE"
            while True:
                line = board.readline().decode(errors="ignore").strip()
                if line == expected:
                    break
                if line in {"BLOCKED", "ERROR"}:
                    raise RuntimeError(f"Arduino stopped: {line}")

In [ ]:
# 改成你的端口再取消注释，例如 Windows 是 "COM4"，macOS/Linux 类似 "/dev/ttyUSB0"。
# send_to_arduino("COM4", result)
print("要下发就填好端口号，然后取消上面那行的注释。")